方案 1 ：也即official baseline

In [4]:
# ============================================
# HOPE-EXP — SIMPLE LR BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_Test_unlabeled.jsonl
# - Model: LogisticRegression() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

OUT_DIR = os.path.join(INPUT_DIR, "baseline_lr_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
     'Hopelessness',
     'Not Hope',
     'Realistic Hope',
     'Sarcastic Hope',
     'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
vec = TfidfVectorizer()  # default params (simple baseline)
Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with LR default params
# -------------------------
clf_A = LogisticRegression()  # default params
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + LR default params
# -------------------------
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

clf_B = OneVsRestClassifier(LogisticRegression())  # default params
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory


/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/sklearn/linear_model/_logistic.py:460: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. of ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


✅ Wrote prediction JSONL!


方案 2（最推荐）：LinearSVC 文本分类里通常比 LR 更强

新增 import

In [5]:
from sklearn.svm import LinearSVC

In [6]:
# ============================================
# HOPE-EXP — SIMPLE LinearSVC BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: LinearSVC() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
     'Hopelessness',
     'Not Hope',
     'Realistic Hope',
     'Sarcastic Hope',
     'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
vec = TfidfVectorizer()  # default params (simple baseline)
Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with LinearSVC default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

clf_A = LinearSVC()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + LinearSVC default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

clf_B = OneVsRestClassifier(LinearSVC())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set t

✅ Wrote prediction JSONL!


方案3：Random Forest（非线性模型）
如果你想尝试树模型。

import 导入模型

In [7]:
from sklearn.ensemble import RandomForestClassifier

In [8]:
# ============================================
# HOPE-EXP — SIMPLE RandomForest BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: RandomForestClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_RandomForest_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
     'Hopelessness',
     'Not Hope',
     'Realistic Hope',
     'Sarcastic Hope',
     'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
vec = TfidfVectorizer()  # default params (simple baseline)
Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with RandomForest default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()
# clf_A = LinearSVC()
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()
clf_A = RandomForestClassifier()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + RandomForest default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

clf_B = OneVsRestClassifier(RandomForestClassifier())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


⚠️ 但注意 RandomForest 在高维 TF-IDF 上通常不如 SVM / LR。

import 导入

In [9]:
from sklearn.naive_bayes import MultinomialNB

In [10]:
# ============================================
# HOPE-EXP — SIMPLE MultinomialNB BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: MultinomialNB() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_MultinomialNB_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
     'Hopelessness',
     'Not Hope',
     'Realistic Hope',
     'Sarcastic Hope',
     'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
vec = TfidfVectorizer()  # default params (simple baseline)
Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with MultinomialNB default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
clf_A = MultinomialNB()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + MultinomialNB default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
clf_B = OneVsRestClassifier(MultinomialNB())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


In [11]:
# ============================================
# HOPE-EXP — SIMPLE LinearSVC BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: LinearSVC() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_TF-IDF_trick_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
          'Hopelessness',
          'Not Hope',
          'Realistic Hope',
          'Sarcastic Hope',
          'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）
# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
# vec = TfidfVectorizer()  # default params (simple baseline)
vec = TfidfVectorizer(
    ngram_range=(1,2),
    min_df=2,
    max_df=0.9
)
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with LinearSVC default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
clf_A = LinearSVC()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + LinearSVC default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
clf_B = OneVsRestClassifier(LinearSVC())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set the value of `dual` explicitly to suppress the warning.
  warnings.warn(
/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/sklearn/svm/_classes.py:32: FutureWarning: The default value of `dual` will change from `True` to `'auto'` in 1.5. Set t

✅ Wrote prediction JSONL!


很好 👍 既然你已经有 Logistic Regression、SVM、RandomForest、Naive Bayes，我再给你补充 几种常见且适合 TF-IDF 文本分类的机器学习方法。这些都可以直接替换 LogisticRegression() 使用。

我会给你 import + 两个任务的替换代码。

In [12]:
from sklearn.ensemble import GradientBoostingClassifier

In [13]:
# ============================================
# HOPE-EXP — SIMPLE GradientBoostingClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: GradientBoostingClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_GradientBoosting_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
      'Hopelessness',
      'Not Hope',
      'Realistic Hope',
      'Sarcastic Hope',
      'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）
# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with GradientBoostingClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
clf_A = GradientBoostingClassifier()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + GradientBoostingClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
clf_B = OneVsRestClassifier(GradientBoostingClassifier())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


2️⃣ Extra Trees（极端随机森林）

比 RandomForest 更随机。

import导入

In [14]:
from sklearn.ensemble import ExtraTreesClassifier

In [15]:
# ============================================
# HOPE-EXP — SIMPLE ExtraTreesClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: ExtraTreesClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_ExtraTrees_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
      'Hopelessness',
      'Not Hope',
      'Realistic Hope',
      'Sarcastic Hope',
      'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with ExtraTreesClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
clf_A = ExtraTreesClassifier()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + ExtraTreesClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
clf_B = OneVsRestClassifier(ExtraTreesClassifier())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


import 导入

In [16]:
from sklearn.neighbors import KNeighborsClassifier

In [17]:
# ============================================
# HOPE-EXP — SIMPLE KNeighborsClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: KNeighborsClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_KNeighbors_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
      'Hopelessness',
      'Not Hope',
      'Realistic Hope',
      'Sarcastic Hope',
      'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。
Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with KNeighborsClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
clf_A = KNeighborsClassifier()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + KNeighborsClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
clf_B = OneVsRestClassifier(KNeighborsClassifier())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

Exception ignored on calling ctypes callback function: <function ThreadpoolController._find_libraries_with_dl_iterate_phdr.<locals>.match_library_callback at 0x79e3c47b1670>
Traceback (most recent call last):
  File "/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/threadpoolctl.py", line 1005, in match_library_callback
    self._make_controller_from_path(filepath)
  File "/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/threadpoolctl.py", line 1175, in _make_controller_from_path
    lib_controller = controller_class(
  File "/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/threadpoolctl.py", line 114, in __init__
    self.dynlib = ctypes.CDLL(filepath, mode=_RTLD_NOLOAD)
  File "/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/ctypes/__init__.py", line 373, in __init__
    self._handle = _dlopen(self._name, mode)
OSError: /home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/libmkl_rt.so.2: can

✅ Wrote prediction JSONL!


In [18]:
from sklearn.linear_model import RidgeClassifier

In [19]:
# ============================================
# HOPE-EXP — SIMPLE RidgeClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: RidgeClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_RidgeClassifier_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
      'Hopelessness',
      'Not Hope',
      'Realistic Hope',
      'Sarcastic Hope',
      'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with RidgeClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
clf_A = RidgeClassifier()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + RidgeClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
clf_B = OneVsRestClassifier(RidgeClassifier())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


优点：

速度快

对文本数据很好

5️⃣ Passive Aggressive Classifier ⭐推荐

这是 专门为文本分类设计的线性模型。

import导入

In [20]:
from sklearn.linear_model import PassiveAggressiveClassifier

In [21]:
# ============================================
# HOPE-EXP — SIMPLE PassiveAggressiveClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: PassiveAggressiveClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_PassiveAggressiveClassifier_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
      'Hopelessness',
      'Not Hope',
      'Realistic Hope',
      'Sarcastic Hope',
      'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with PassiveAggressiveClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
clf_A = PassiveAggressiveClassifier()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + PassiveAggressiveClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


特点：

非常适合 稀疏 TF-IDF

训练速度快

6️⃣ SGDClassifier（线性模型）⭐强烈推荐

很多 NLP baseline 都用这个。

In [1]:
from sklearn.linear_model import SGDClassifier

In [2]:
# ============================================
# HOPE-EXP — SIMPLE SGDClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: SGDClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_SGDClassifier_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
     'Hopelessness',
     'Not Hope',
     'Realistic Hope',
     'Sarcastic Hope',
     'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with SGDClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
clf_A = SGDClassifier()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + SGDClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
clf_B = OneVsRestClassifier(SGDClassifier())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


优点：

支持大规模数据

效果接近 SVM

很好 👍 既然你是在做 TF-IDF + 传统机器学习 baseline，我再给你继续推荐一些 可以直接替换 LogisticRegression() 的模型。下面这些 在 sklearn 中也常见，有些在文本任务中表现也不错。

我继续按 import 导入+ Task A 代码+ Task B 代码给你。

1️⃣ Decision Tree（决策树）

import 导入

In [3]:
from sklearn.tree import DecisionTreeClassifier

In [4]:
# ============================================
# HOPE-EXP — SIMPLE DecisionTreeClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: DecisionTreeClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_DecisionTree_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
      'Hopelessness',
      'Not Hope',
      'Realistic Hope',
      'Sarcastic Hope',
      'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with DecisionTreeClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
# clf_A = SGDClassifier()
clf_A = DecisionTreeClassifier()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + DecisionTreeClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
# clf_B = OneVsRestClassifier(SGDClassifier())
clf_B = OneVsRestClassifier(DecisionTreeClassifier())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


特点：

可解释性强

但对高维文本特征效果一般

2️⃣ AdaBoost

Boosting 方法之一。

import 导入

In [5]:
from sklearn.ensemble import AdaBoostClassifier

In [6]:
# ============================================
# HOPE-EXP — SIMPLE AdaBoostClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: AdaBoostClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_AdaBoost_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
      'Hopelessness',
      'Not Hope',
      'Realistic Hope',
      'Sarcastic Hope',
      'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with AdaBoostClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
# clf_A = SGDClassifier()
# clf_A = DecisionTreeClassifier()
clf_A = AdaBoostClassifier()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + AdaBoostClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
# clf_B = OneVsRestClassifier(SGDClassifier())
# clf_B = OneVsRestClassifier(DecisionTreeClassifier())
clf_B = OneVsRestClassifier(AdaBoostClassifier())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


特点：

能处理非线性

对噪声比较敏感

3️⃣ BaggingClassifier

Bagging 集成方法。

import 导入

In [7]:
from sklearn.ensemble import BaggingClassifier

In [8]:
# ============================================
# HOPE-EXP — SIMPLE BaggingClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: BaggingClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_Bagging_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
      'Hopelessness',
      'Not Hope',
      'Realistic Hope',
      'Sarcastic Hope',
      'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with BaggingClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
# clf_A = SGDClassifier()
# clf_A = DecisionTreeClassifier()
# clf_A = AdaBoostClassifier()
clf_A = BaggingClassifier()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + BaggingClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
# clf_B = OneVsRestClassifier(SGDClassifier())
# clf_B = OneVsRestClassifier(DecisionTreeClassifier())
# clf_B = OneVsRestClassifier(AdaBoostClassifier())
clf_B = OneVsRestClassifier(BaggingClassifier())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


特点：

降低过拟合

稳定性好

4️⃣ Bernoulli Naive Bayes

如果 TF-IDF 转成 binary 特征会很好。

import 导入

In [9]:
from sklearn.naive_bayes import BernoulliNB

In [10]:
# ============================================
# HOPE-EXP — SIMPLE BernoulliNB BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: BernoulliNB() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_BernoulliNaiveBayes_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
     'Hopelessness',
     'Not Hope',
     'Realistic Hope',
     'Sarcastic Hope',
     'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with BernoulliNB default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
# clf_A = SGDClassifier()
# clf_A = DecisionTreeClassifier()
# clf_A = AdaBoostClassifier()
# clf_A = BaggingClassifier()
clf_A = BernoulliNB()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + BernoulliNB default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
# clf_B = OneVsRestClassifier(SGDClassifier())
# clf_B = OneVsRestClassifier(DecisionTreeClassifier())
# clf_B = OneVsRestClassifier(AdaBoostClassifier())
# clf_B = OneVsRestClassifier(BaggingClassifier())
clf_B = OneVsRestClassifier(BernoulliNB())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


特点：

对短文本效果好

速度极快

5️⃣ Complement Naive Bayes ⭐（文本任务常用）

这是专门针对 文本分类改进的 Naive Bayes。

import 导入

In [11]:
from sklearn.naive_bayes import ComplementNB

In [12]:
# ============================================
# HOPE-EXP — SIMPLE ComplementNB BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: ComplementNB() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_ComplementNB_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
     'Hopelessness',
     'Not Hope',
     'Realistic Hope',
     'Sarcastic Hope',
     'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with ComplementNB default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
# clf_A = SGDClassifier()
# clf_A = DecisionTreeClassifier()
# clf_A = AdaBoostClassifier()
# clf_A = BaggingClassifier()
# clf_A = BernoulliNB()
clf_A = ComplementNB()
clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + ComplementNB default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
# clf_B = OneVsRestClassifier(SGDClassifier())
# clf_B = OneVsRestClassifier(DecisionTreeClassifier())
# clf_B = OneVsRestClassifier(AdaBoostClassifier())
# clf_B = OneVsRestClassifier(BaggingClassifier())
# clf_B = OneVsRestClassifier(BernoulliNB())
clf_B = OneVsRestClassifier(ComplementNB())
clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


优点：

对类别不平衡更稳定

NLP 论文常用

6️⃣ Gaussian Naive Bayes

import 导入

In [13]:
from sklearn.naive_bayes import GaussianNB

⚠️ 但注意
GaussianNB 不支持 sparse matrix。

In [14]:
# ============================================
# HOPE-EXP — SIMPLE GaussianNB BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: GaussianNB() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_GaussianNB_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
     'Hopelessness',
     'Not Hope',
     'Realistic Hope',
     'Sarcastic Hope',
     'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])
# 需要：
Xtr_dense = Xtr.toarray()
Xdv_dense = Xdv.toarray()

# -------------------------
# Task A: primary_label (multiclass) with GaussianNB default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
# clf_A = SGDClassifier()
# clf_A = DecisionTreeClassifier()
# clf_A = AdaBoostClassifier()
# clf_A = BaggingClassifier()
# clf_A = BernoulliNB()
# clf_A = ComplementNB()
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# 然后：
clf_A = GaussianNB()
y = df_train["primary_label"].astype(str).values
clf_A.fit(Xtr_dense, y)
pred_A = clf_A.predict(Xdv_dense).tolist()


# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + GaussianNB default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])
y = Ytr

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
# clf_B = OneVsRestClassifier(SGDClassifier())
# clf_B = OneVsRestClassifier(DecisionTreeClassifier())
# clf_B = OneVsRestClassifier(AdaBoostClassifier())
# clf_B = OneVsRestClassifier(BaggingClassifier())
# clf_B = OneVsRestClassifier(BernoulliNB())
# clf_B = OneVsRestClassifier(ComplementNB())
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

clf_B = OneVsRestClassifier(GaussianNB())
clf_B.fit(Xtr_dense, y)
pred_B_bin = clf_B.predict(Xdv_dense)
# 一般 不推荐用于 TF-IDF。

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


7️⃣ HistGradientBoosting（新一代GBDT）

import 导入

In [15]:
from sklearn.ensemble import HistGradientBoostingClassifier

In [17]:
# ============================================
# HOPE-EXP — SIMPLE HistGradientBoostingClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: HistGradientBoostingClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_HistGradientBoosting_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
      'Hopelessness',
      'Not Hope',
      'Realistic Hope',
      'Sarcastic Hope',
      'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])
Xtr_dense = Xtr.toarray()
Xdv_dense = Xdv.toarray()

# -------------------------
# Task A: primary_label (multiclass) with HistGradientBoostingClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
# clf_A = SGDClassifier()
# clf_A = DecisionTreeClassifier()
# clf_A = AdaBoostClassifier()
# clf_A = BaggingClassifier()
# clf_A = BernoulliNB()
# clf_A = ComplementNB()
clf_A = HistGradientBoostingClassifier()
clf_A.fit(Xtr_dense, df_train["primary_label"].astype(str).values)
pred_A = clf_A.predict(Xdv_dense).tolist()

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + HistGradientBoostingClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
# clf_B = OneVsRestClassifier(SGDClassifier())
# clf_B = OneVsRestClassifier(DecisionTreeClassifier())
# clf_B = OneVsRestClassifier(AdaBoostClassifier())
# clf_B = OneVsRestClassifier(BaggingClassifier())
# clf_B = OneVsRestClassifier(BernoulliNB())
# clf_B = OneVsRestClassifier(ComplementNB())
clf_B = OneVsRestClassifier(HistGradientBoostingClassifier())
clf_B.fit(Xtr_dense, Ytr)
pred_B_bin = clf_B.predict(Xdv_dense)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


特点：

比传统 GBDT 快

但对 sparse 文本仍不理想

使用了sklearn以外的依赖包进行分类

很好 👍 如果你想在 sklearn 之外尝试更多机器学习模型，下面给你 10 个常见且强大的模型（很多 Kaggle / NLP 竞赛都会用）。这些模型基本都可以替换你代码中的 LogisticRegression()。

我会给你 库名称 + 简要说明 + 代码示例。

1️⃣ XGBoost ⭐ 非常常用

库：xgboost

优点：

非常成熟

Kaggle 常用

对结构化特征效果很好

安装

In [18]:
!pip install xgboost

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.6/223.6 MB 1.9 MB/s eta 0:00:0000:0100:03
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 293.6/293.6 MB 4.0 MB/s eta 0:00:0000:0100:03


import 导入

In [19]:
from xgboost import XGBClassifier

In [21]:
# ============================================
# HOPE-EXP — SIMPLE XGBClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: XGBClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_XGBClassifier_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
      'Hopelessness',
      'Not Hope',
      'Realistic Hope',
      'Sarcastic Hope',
      'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])
Xtr_dense = Xtr.toarray()
Xdv_dense = Xdv.toarray()

# -------------------------
# Task A: primary_label (multiclass) with XGBoost default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
# clf_A = SGDClassifier()
# clf_A = DecisionTreeClassifier()
# clf_A = AdaBoostClassifier()
# clf_A = BaggingClassifier()
# clf_A = BernoulliNB()
# clf_A = ComplementNB()
# clf_A = HistGradientBoostingClassifier()
clf_A = XGBClassifier()
y_train = le.fit_transform(df_train["primary_label"].astype(str))
clf_A.fit(Xtr_dense, y_train)
pred_A = clf_A.predict(Xdv_dense).tolist()
# 如果需要还原成原始标签
pred_A = le.inverse_transform(pred_A)

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + XGBoost default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
# clf_B = OneVsRestClassifier(SGDClassifier())
# clf_B = OneVsRestClassifier(DecisionTreeClassifier())
# clf_B = OneVsRestClassifier(AdaBoostClassifier())
# clf_B = OneVsRestClassifier(BaggingClassifier())
# clf_B = OneVsRestClassifier(BernoulliNB())
# clf_B = OneVsRestClassifier(ComplementNB())
# clf_B = OneVsRestClassifier(HistGradientBoostingClassifier())
clf_B = OneVsRestClassifier(XGBClassifier())
clf_B.fit(Xtr_dense, Ytr)
pred_B_bin = clf_B.predict(Xdv_dense)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


更稳定的 XGBoost 写法（推荐）

建议显式指定参数：

In [ ]:
# ============================================
# HOPE-EXP — SIMPLE XGBoost BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: XGBClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_XGBClassifier_parameter_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
    'Hopelessness',
    'Not Hope',
    'Realistic Hope',
    'Sarcastic Hope',
    'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])
Xtr_dense = Xtr.toarray()
Xdv_dense = Xdv.toarray()
# -------------------------
# Task A: primary_label (multiclass) with XGBClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
# clf_A = SGDClassifier()
# clf_A = DecisionTreeClassifier()
# clf_A = AdaBoostClassifier()
# clf_A = BaggingClassifier()
# clf_A = BernoulliNB()
# clf_A = ComplementNB()
# clf_A = HistGradientBoostingClassifier()
y_train = le.fit_transform(df_train["primary_label"].astype(str))
clf_A = XGBClassifier( objective="multi:softmax",
                       num_class=len(le.classes_),
                       eval_metric="mlogloss")
clf_A.fit(Xtr_dense, y_train)
pred_A = clf_A.predict(Xdv_dense).tolist()
# 如果需要还原成原始标签
pred_A = le.inverse_transform(pred_A)

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + XGBClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
# clf_B = OneVsRestClassifier(SGDClassifier())
# clf_B = OneVsRestClassifier(DecisionTreeClassifier())
# clf_B = OneVsRestClassifier(AdaBoostClassifier())
# clf_B = OneVsRestClassifier(BaggingClassifier())
# clf_B = OneVsRestClassifier(BernoulliNB())
# clf_B = OneVsRestClassifier(ComplementNB())
# clf_B = OneVsRestClassifier(HistGradientBoostingClassifier())
clf_B = OneVsRestClassifier(XGBClassifier())
clf_B.fit(Xtr_dense, Ytr)
pred_B_bin = clf_B.predict(Xdv_dense)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

额外建议（非常重要）

你的输入是 TF-IDF → Dense → XGBoost

通常 不要转 dense，否则非常占内存。

XGBoost 可以直接吃 sparse matrix：

In [23]:
# ============================================
# HOPE-EXP — SIMPLE XGBClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: XGBClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_XGBClassifier_parameter_sparse matrix_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
    'Hopelessness',
    'Not Hope',
    'Realistic Hope',
    'Sarcastic Hope',
    'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])
# Xtr_dense = Xtr.toarray()
# Xdv_dense = Xdv.toarray()

# -------------------------
# Task A: primary_label (multiclass) with XGBClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
# clf_A = SGDClassifier()
# clf_A = DecisionTreeClassifier()
# clf_A = AdaBoostClassifier()
# clf_A = BaggingClassifier()
# clf_A = BernoulliNB()
# clf_A = ComplementNB()
# clf_A = HistGradientBoostingClassifier()
y_train = le.fit_transform(df_train["primary_label"].astype(str))
clf_A = XGBClassifier( objective="multi:softmax",
                       num_class=len(le.classes_),
                       eval_metric="mlogloss")
# clf_A.fit(Xtr_dense, y_train)
clf_A.fit(Xtr, y_train)
# pred_A = clf_A.predict(Xdv_dense).tolist()
pred_A = clf_A.predict(Xdv).tolist()
# 如果需要还原成原始标签
pred_A = le.inverse_transform(pred_A)

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + XGBClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
# clf_B = OneVsRestClassifier(SGDClassifier())
# clf_B = OneVsRestClassifier(DecisionTreeClassifier())
# clf_B = OneVsRestClassifier(AdaBoostClassifier())
# clf_B = OneVsRestClassifier(BaggingClassifier())
# clf_B = OneVsRestClassifier(BernoulliNB())
# clf_B = OneVsRestClassifier(ComplementNB())
# clf_B = OneVsRestClassifier(HistGradientBoostingClassifier())
clf_B = OneVsRestClassifier(XGBClassifier())
# clf_B.fit(Xtr_dense, Ytr)
clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv_dense)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


2️⃣ LightGBM ⭐ 非常推荐

库：lightgbm
优点：

训练速度快

内存占用低

对高维特征友好

安装

In [24]:
!pip install lightgbm

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 7.0 MB/s eta 0:00:00a 0:00:01


import 导入

In [25]:
from lightgbm import LGBMClassifier

In [26]:
# ============================================
# HOPE-EXP — SIMPLE LGBMClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: LGBMClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_LightGBM_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
     'Hopelessness',
     'Not Hope',
     'Realistic Hope',
     'Sarcastic Hope',
     'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])
# Xtr_dense = Xtr.toarray()
# Xdv_dense = Xdv.toarray()

# -------------------------
# Task A: primary_label (multiclass) with LGBMClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
# clf_A = SGDClassifier()
# clf_A = DecisionTreeClassifier()
# clf_A = AdaBoostClassifier()
# clf_A = BaggingClassifier()
# clf_A = BernoulliNB()
# clf_A = ComplementNB()
# clf_A = HistGradientBoostingClassifier()
y_train = le.fit_transform(df_train["primary_label"].astype(str))
# 使用
clf_A = LGBMClassifier()
# clf_A = XGBClassifier( objective="multi:softmax",
#                        num_class=len(le.classes_),
#                        eval_metric="mlogloss")
# clf_A.fit(Xtr_dense, y_train)
clf_A.fit(Xtr, y_train)
# pred_A = clf_A.predict(Xdv_dense).tolist()
pred_A = clf_A.predict(Xdv).tolist()
# 如果需要还原成原始标签
pred_A = le.inverse_transform(pred_A)

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + LGBMClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
# clf_B = OneVsRestClassifier(SGDClassifier())
# clf_B = OneVsRestClassifier(DecisionTreeClassifier())
# clf_B = OneVsRestClassifier(AdaBoostClassifier())
# clf_B = OneVsRestClassifier(BaggingClassifier())
# clf_B = OneVsRestClassifier(BernoulliNB())
# clf_B = OneVsRestClassifier(ComplementNB())
# clf_B = OneVsRestClassifier(HistGradientBoostingClassifier())
# clf_B = OneVsRestClassifier(XGBClassifier())
clf_B = OneVsRestClassifier(LGBMClassifier())
# clf_B.fit(Xtr_dense, Ytr)
clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv_dense)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.048620 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 137913
[LightGBM] [Info] Number of data points in the train set: 4857, number of used features: 3891
[LightGBM] [Info] Start training from score -1.645493
[LightGBM] [Info] Start training from score -2.450305
[LightGBM] [Info] Start training from score -1.243949
[LightGBM] [Info] Start training from score -1.935668
[LightGBM] [Info] Start training from score -1.937096
[LightGBM] [Info] Start training from score -1.937096
[LightGBM] [Info] Number of positive: 2438, number of negative: 2419
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.035353 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 137913
[LightGBM] [Info] Number of data points in the train set: 4857, number of used features: 3891
[LightGBM] [Info] [bina

3️⃣ CatBoost ⭐ 文本处理友好

库：catboost

优点：

对 categorical 特征很好

默认参数就很强

安装

In [27]:
!pip install catboost

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 97.1/97.1 MB 3.5 MB/s eta 0:00:0000:0100:01
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/30/33/cc27211d2ffeee4fd7402dca137b6e8a83f6dcae3d4be8d0ad5068555561/matplotlib-3.7.5-cp38-cp38-manylinux_2_12_x86_64.manylinux2010_x86_64.whl (9.2 MB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.9/9.9 MB 3.6 MB/s eta 0:00:0000:0100:01
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/8e/71/7f20855592cc929bc206810432b991ec4c702dc26b0567b132e52c85536f/contourpy-1.1.1-cp38-cp38-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (301 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/e7/05/c19819d5e3d95294a6f5947fb9b9629efb316b96de511b418c53d245aae6/cycler-0.12.1-py3-none-any.whl (8.3 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/a5/0e/b6314a09a4d561aaa7e09de43fa700917be91e701f07df6178865962666c/fonttools-4.57.0-cp38-cp38-manylinux_2_17_x86_64.manylin

import 导入

In [28]:
from catboost import CatBoostClassifier

In [29]:
# ============================================
# HOPE-EXP — SIMPLE CatBoostClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: CatBoostClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_CatBoost_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
    'Hopelessness',
    'Not Hope',
    'Realistic Hope',
    'Sarcastic Hope',
    'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])
# Xtr_dense = Xtr.toarray()
# Xdv_dense = Xdv.toarray()

# -------------------------
# Task A: primary_label (multiclass) with CatBoostClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
# clf_A = SGDClassifier()
# clf_A = DecisionTreeClassifier()
# clf_A = AdaBoostClassifier()
# clf_A = BaggingClassifier()
# clf_A = BernoulliNB()
# clf_A = ComplementNB()
# clf_A = HistGradientBoostingClassifier()
y_train = le.fit_transform(df_train["primary_label"].astype(str))
# 使用
# clf_A = LGBMClassifier()
clf_A = CatBoostClassifier(verbose=0)
# clf_A = XGBClassifier( objective="multi:softmax",
#                        num_class=len(le.classes_),
#                        eval_metric="mlogloss")
# clf_A.fit(Xtr_dense, y_train)
clf_A.fit(Xtr, y_train)
# pred_A = clf_A.predict(Xdv_dense).tolist()
pred_A = clf_A.predict(Xdv).tolist()
# 如果需要还原成原始标签
pred_A = le.inverse_transform(pred_A)

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + CatBoostClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
# clf_B = OneVsRestClassifier(SGDClassifier())
# clf_B = OneVsRestClassifier(DecisionTreeClassifier())
# clf_B = OneVsRestClassifier(AdaBoostClassifier())
# clf_B = OneVsRestClassifier(BaggingClassifier())
# clf_B = OneVsRestClassifier(BernoulliNB())
# clf_B = OneVsRestClassifier(ComplementNB())
# clf_B = OneVsRestClassifier(HistGradientBoostingClassifier())
# clf_B = OneVsRestClassifier(XGBClassifier())
# clf_B = OneVsRestClassifier(LGBMClassifier())
clf_B = OneVsRestClassifier(CatBoostClassifier(verbose=0))
# clf_B.fit(Xtr_dense, Ytr)
clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv_dense)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/sklearn/preprocessing/_label.py:153: DataConversionWarning: A column-vector y was passed when a 1d array was expected. Please change the shape of y to (n_samples, ), for example using ravel().
  y = column_or_1d(y, warn=True)


✅ Wrote prediction JSONL!


4️⃣ NGBoost

库：ngboost

特点：

Probabilistic gradient boosting

可以输出概率分布

安装

In [5]:
!pip install -U ngboost==0.4

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Attempting uninstall: ngboost
    Found existing installation: ngboost 0.3.13
    Uninstalling ngboost-0.3.13:
      Successfully uninstalled ngboost-0.3.13


import 导入

In [4]:
from ngboost import NGBClassifier
from ngboost.distns import k_categorical

In [5]:
# ============================================
# HOPE-EXP — SIMPLE NGBClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: NGBClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_NGBoost_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
    'Hopelessness',
    'Not Hope',
    'Realistic Hope',
    'Sarcastic Hope',
    'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])
# Xtr_dense = Xtr.toarray()
# Xdv_dense = Xdv.toarray()

# -------------------------
# Task A: primary_label (multiclass) with NGBClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
# clf_A = SGDClassifier()
# clf_A = DecisionTreeClassifier()
# clf_A = AdaBoostClassifier()
# clf_A = BaggingClassifier()
# clf_A = BernoulliNB()
# clf_A = ComplementNB()
# clf_A = HistGradientBoostingClassifier()
y_train = le.fit_transform(df_train["primary_label"].astype(str))
# 使用
# clf_A = LGBMClassifier()
# clf_A = CatBoostClassifier(verbose=0)
# NGBoost默认是二分类，而你的任务是 6分类。
clf_A = NGBClassifier(Dist=k_categorical(len(set(y_train))))
# clf_A = XGBClassifier( objective="multi:softmax",
#                        num_class=len(le.classes_),
#                        eval_metric="mlogloss")
# clf_A.fit(Xtr_dense, y_train)
clf_A.fit(Xtr, y_train)
# pred_A = clf_A.predict(Xdv_dense).tolist()
pred_A = clf_A.predict(Xdv).tolist()
# 如果需要还原成原始标签
pred_A = le.inverse_transform(pred_A)

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + NGBClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
# clf_B = OneVsRestClassifier(SGDClassifier())
# clf_B = OneVsRestClassifier(DecisionTreeClassifier())
# clf_B = OneVsRestClassifier(AdaBoostClassifier())
# clf_B = OneVsRestClassifier(BaggingClassifier())
# clf_B = OneVsRestClassifier(BernoulliNB())
# clf_B = OneVsRestClassifier(ComplementNB())
# clf_B = OneVsRestClassifier(HistGradientBoostingClassifier())
# clf_B = OneVsRestClassifier(XGBClassifier())
# clf_B = OneVsRestClassifier(LGBMClassifier())
# clf_B = OneVsRestClassifier(CatBoostClassifier(verbose=0))
clf_B = OneVsRestClassifier(NGBClassifier())
# clf_B.fit(Xtr_dense, Ytr)
clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv_dense)
pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

[iter 0] loss=1.7251 val_loss=0.0000 scale=2.0000 norm=14.4721
[iter 100] loss=1.0172 val_loss=0.0000 scale=1.0000 norm=3.9577
[iter 200] loss=0.8919 val_loss=0.0000 scale=1.0000 norm=3.5624
[iter 300] loss=0.8209 val_loss=0.0000 scale=1.0000 norm=3.3565
[iter 400] loss=0.7725 val_loss=0.0000 scale=1.0000 norm=3.2187
[iter 0] loss=0.6931 val_loss=0.0000 scale=2.0000 norm=4.0000
[iter 100] loss=0.5784 val_loss=0.0000 scale=2.0000 norm=3.6521
[iter 200] loss=0.5425 val_loss=0.0000 scale=1.0000 norm=1.7796
[iter 300] loss=0.5225 val_loss=0.0000 scale=2.0000 norm=3.5054
[iter 400] loss=0.5078 val_loss=0.0000 scale=2.0000 norm=3.4669
[iter 0] loss=0.3795 val_loss=0.0000 scale=4.0000 norm=8.0000
[iter 100] loss=0.2879 val_loss=0.0000 scale=1.0000 norm=1.7551
[iter 200] loss=0.2699 val_loss=0.0000 scale=1.0000 norm=1.7342
[iter 300] loss=0.2608 val_loss=0.0000 scale=1.0000 norm=1.7228
[iter 400] loss=0.2527 val_loss=0.0000 scale=1.0000 norm=1.7067
[iter 0] loss=0.3783 val_loss=0.0000 scale=1.

5️⃣ TabNet（深度学习表格模型）

库：pytorch-tabnet

优点：

使用 attention

适合结构化数据

安装

In [6]:
!pip install pytorch-tabnet

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/a9/71/45aac46b75742e08d2d6f9fc2b612223b5e36115b8b2ed673b23c21b5387/torch-2.4.1-cp38-cp38-manylinux1_x86_64.whl (797.1 MB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/b9/f8/feced7779d755758a52d1f6635d990b8d98dc0a29fa568bbe0625f18fdf3/filelock-3.16.1-py3-none-any.whl (16 kB)
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.2/6.2 MB 7.9 MB/s eta 0:00:00a 0:00:01
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/a8/05/9d4f9b78ead6b2661d6e8ea772e111fc4a9fbd866ad0c81906c11206b55e/networkx-3.1-py3-none-any.whl (2.1 MB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/56/53/eb690efa8513166adef3e0669afd31e95ffde69fb3c52ec2ac7223ed6018/fsspec-2025.3.0-py3-none-any.whl (193 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/b6/9f/c64c03f49d6fbc56196664d05dba14e3a561038a81a638eeb47f4d4cfd48/nvidia_cuda_nvrtc_cu12-12.1.105-py3-none-manylinux

import 导入

In [1]:
from pytorch_tabnet.tab_model import TabNetClassifier
from sklearn.decomposition import TruncatedSVD

⚠️ 注意
需要 dense array

In [3]:
# ============================================
# HOPE-EXP — SIMPLE TabNetClassifier BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: TabNetClassifier() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.svm import LinearSVC
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.preprocessing import LabelEncoder
le = LabelEncoder()
# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

# OUT_DIR = os.path.join(INPUT_DIR, "baseline_LinearSVC_submission")
OUT_DIR = os.path.join(INPUT_DIR, "baseline_TabNet_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
     'Hopelessness',
     'Not Hope',
     'Realistic Hope',
     'Sarcastic Hope',
     'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# 额外建议（比赛常见 trick）

# 如果你想 稍微提升一点 baseline 分数，可以把 TF-IDF 改成：
vec = TfidfVectorizer()  # default params (simple baseline)
# vec = TfidfVectorizer(
#     ngram_range=(1,2),
#     min_df=2,
#     max_df=0.9
# )
# 通常能涨 1-3%。

Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])
# Xtr_dense = Xtr.toarray()
# Xdv_dense = Xdv.toarray()

# 如果你一定要用 TabNet
# 方法 1：降维（必须）

# 先降维：

svd = TruncatedSVD(n_components=300)
Xtr_reduced = svd.fit_transform(Xtr)
Xdv_reduced = svd.transform(Xdv)

# 训练速度会提升 100 倍。
# -------------------------
# Task A: primary_label (multiclass) with TabNetClassifier default params
# -------------------------
### Task A 修改
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)
# pred_A = clf_A.predict(Xdv).tolist()

# clf_A = LinearSVC()
# clf_A = RandomForestClassifier()
# clf_A = MultinomialNB()
# clf_A = LinearSVC()
# clf_A = GradientBoostingClassifier()
# clf_A = ExtraTreesClassifier()
# clf_A = KNeighborsClassifier()
# clf_A = RidgeClassifier()
# clf_A = PassiveAggressiveClassifier()
# clf_A = SGDClassifier()
# clf_A = DecisionTreeClassifier()
# clf_A = AdaBoostClassifier()
# clf_A = BaggingClassifier()
# clf_A = BernoulliNB()
# clf_A = ComplementNB()
# clf_A = HistGradientBoostingClassifier()
y_train = le.fit_transform(df_train["primary_label"].astype(str))
# 使用
clf_A = TabNetClassifier()
# clf_A = LGBMClassifier()
# clf_A = CatBoostClassifier(verbose=0)
# NGBoost默认是二分类，而你的任务是 6分类。
# clf_A = NGBClassifier(Dist=k_categorical(len(set(y_train))))
# clf_A = XGBClassifier( objective="multi:softmax",
#                        num_class=len(le.classes_),
#                        eval_metric="mlogloss")

# 方法 2：减少 epoch
# 方法 3：加入 early stopping
clf_A.fit(Xtr_reduced, y_train, eval_set=[(Xtr_reduced, y_train)],
    patience=10, max_epochs=20)
# clf_A.fit(Xtr, y_train)
pred_A = clf_A.predict(Xdv_reduced).tolist()
# pred_A = clf_A.predict(Xdv).tolist()
# 如果需要还原成原始标签
pred_A = le.inverse_transform(pred_A)

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + TabNetClassifier default params
# -------------------------
## Task B 修改
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)

# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(RandomForestClassifier())
# clf_B = OneVsRestClassifier(MultinomialNB())
# clf_B = OneVsRestClassifier(LinearSVC())
# clf_B = OneVsRestClassifier(GradientBoostingClassifier())
# clf_B = OneVsRestClassifier(ExtraTreesClassifier())
# clf_B = OneVsRestClassifier(KNeighborsClassifier())
# clf_B = OneVsRestClassifier(RidgeClassifier())
# clf_B = OneVsRestClassifier(PassiveAggressiveClassifier())
# clf_B = OneVsRestClassifier(SGDClassifier())
# clf_B = OneVsRestClassifier(DecisionTreeClassifier())
# clf_B = OneVsRestClassifier(AdaBoostClassifier())
# clf_B = OneVsRestClassifier(BaggingClassifier())
# clf_B = OneVsRestClassifier(BernoulliNB())
# clf_B = OneVsRestClassifier(ComplementNB())
# clf_B = OneVsRestClassifier(HistGradientBoostingClassifier())
# clf_B = OneVsRestClassifier(XGBClassifier())
# clf_B = OneVsRestClassifier(LGBMClassifier())
# clf_B = OneVsRestClassifier(CatBoostClassifier(verbose=0))
# clf_B = OneVsRestClassifier(NGBClassifier())
clf_B = OneVsRestClassifier(TabNetClassifier())
clf_B.fit(Xtr_reduced, Ytr)
# clf_B.fit(Xtr, Ytr)
pred_B_bin = clf_B.predict(Xdv_reduced)
# pred_B_bin = clf_B.predict(Xdv)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")


epoch 0  | loss: 2.04814 | val_0_accuracy: 0.14412 |  0:00:00s
epoch 1  | loss: 1.86103 | val_0_accuracy: 0.14412 |  0:00:01s
epoch 2  | loss: 1.80563 | val_0_accuracy: 0.28433 |  0:00:01s
epoch 3  | loss: 1.77542 | val_0_accuracy: 0.28824 |  0:00:01s
epoch 4  | loss: 1.73912 | val_0_accuracy: 0.28824 |  0:00:01s
epoch 5  | loss: 1.70134 | val_0_accuracy: 0.28824 |  0:00:02s
epoch 6  | loss: 1.67547 | val_0_accuracy: 0.28763 |  0:00:02s
epoch 7  | loss: 1.62949 | val_0_accuracy: 0.34713 |  0:00:02s
epoch 8  | loss: 1.61753 | val_0_accuracy: 0.33272 |  0:00:03s
epoch 9  | loss: 1.58386 | val_0_accuracy: 0.34816 |  0:00:03s
epoch 10 | loss: 1.56166 | val_0_accuracy: 0.28248 |  0:00:03s
epoch 11 | loss: 1.5273  | val_0_accuracy: 0.19292 |  0:00:03s
epoch 12 | loss: 1.50701 | val_0_accuracy: 0.19292 |  0:00:04s
epoch 13 | loss: 1.47519 | val_0_accuracy: 0.19477 |  0:00:04s
epoch 14 | loss: 1.44965 | val_0_accuracy: 0.20033 |  0:00:04s
epoch 15 | loss: 1.42549 | val_0_accuracy: 0.27198 |  0

/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/pytorch_tabnet/callbacks.py:172: UserWarning: Best weights from best epoch are automatically used!
  warnings.warn(wrn_msg)
/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")
/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")
/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


epoch 0  | loss: 0.93634 |  0:00:00s
epoch 1  | loss: 0.80448 |  0:00:00s
epoch 2  | loss: 0.73479 |  0:00:00s
epoch 3  | loss: 0.71145 |  0:00:00s
epoch 4  | loss: 0.70412 |  0:00:00s
epoch 5  | loss: 0.68954 |  0:00:00s
epoch 6  | loss: 0.68188 |  0:00:00s
epoch 7  | loss: 0.66631 |  0:00:01s
epoch 8  | loss: 0.6664  |  0:00:01s
epoch 9  | loss: 0.66672 |  0:00:01s
epoch 10 | loss: 0.65502 |  0:00:01s
epoch 11 | loss: 0.65308 |  0:00:01s
epoch 12 | loss: 0.64856 |  0:00:01s
epoch 13 | loss: 0.65149 |  0:00:01s
epoch 14 | loss: 0.64672 |  0:00:01s
epoch 15 | loss: 0.64443 |  0:00:02s
epoch 16 | loss: 0.64351 |  0:00:02s
epoch 17 | loss: 0.64403 |  0:00:02s
epoch 18 | loss: 0.64636 |  0:00:02s
epoch 19 | loss: 0.63616 |  0:00:02s
epoch 20 | loss: 0.64328 |  0:00:02s
epoch 21 | loss: 0.63519 |  0:00:02s
epoch 22 | loss: 0.63621 |  0:00:02s
epoch 23 | loss: 0.63424 |  0:00:03s
epoch 24 | loss: 0.6349  |  0:00:03s
epoch 25 | loss: 0.63209 |  0:00:03s
epoch 26 | loss: 0.63126 |  0:00:03s
e

/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")
/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


...

epoch 0  | loss: 0.71054 |  0:00:00s
epoch 1  | loss: 0.60478 |  0:00:00s
epoch 2  | loss: 0.57846 |  0:00:00s
epoch 3  | loss: 0.5493  |  0:00:00s
epoch 4  | loss: 0.54007 |  0:00:00s
epoch 5  | loss: 0.53961 |  0:00:00s
epoch 6  | loss: 0.53626 |  0:00:00s
epoch 7  | loss: 0.52717 |  0:00:01s
epoch 8  | loss: 0.53131 |  0:00:01s
epoch 9  | loss: 0.52716 |  0:00:01s
epoch 10 | loss: 0.52372 |  0:00:01s
epoch 11 | loss: 0.52533 |  0:00:01s
epoch 12 | loss: 0.5141  |  0:00:01s
epoch 13 | loss: 0.52165 |  0:00:01s
epoch 14 | loss: 0.51689 |  0:00:02s
epoch 15 | loss: 0.50997 |  0:00:02s
epoch 16 | loss: 0.50366 |  0:00:02s
epoch 17 | loss: 0.51541 |  0:00:02s
epoch 18 | loss: 0.50599 |  0:00:02s
epoch 19 | loss: 0.51008 |  0:00:02s
epoch 20 | loss: 0.50414 |  0:00:02s
epoch 21 | loss: 0.49882 |  0:00:02s
epoch 22 | loss: 0.50396 |  0:00:03s
epoch 23 | loss: 0.49805 |  0:00:06s
epoch 24 | loss: 0.49116 |  0:00:06s
epoch 25 | loss: 0.48754 |  0:00:06s
epoch 26 | loss: 0.48901 |  0:00:06s
e

/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")
/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


epoch 0  | loss: 0.46554 |  0:00:00s
epoch 1  | loss: 0.35502 |  0:00:00s
epoch 2  | loss: 0.35297 |  0:00:00s
epoch 3  | loss: 0.30928 |  0:00:00s
epoch 4  | loss: 0.29702 |  0:00:00s
epoch 5  | loss: 0.28677 |  0:00:00s
epoch 6  | loss: 0.27571 |  0:00:00s
epoch 7  | loss: 0.29075 |  0:00:01s
epoch 8  | loss: 0.27932 |  0:00:01s
epoch 9  | loss: 0.27229 |  0:00:01s
epoch 10 | loss: 0.27243 |  0:00:01s
epoch 11 | loss: 0.27089 |  0:00:01s
epoch 12 | loss: 0.27611 |  0:00:01s
epoch 13 | loss: 0.27503 |  0:00:01s
epoch 14 | loss: 0.27128 |  0:00:02s
epoch 15 | loss: 0.26952 |  0:00:02s
epoch 16 | loss: 0.27233 |  0:00:02s
epoch 17 | loss: 0.26594 |  0:00:02s
epoch 18 | loss: 0.26652 |  0:00:02s
epoch 19 | loss: 0.26727 |  0:00:02s
epoch 20 | loss: 0.26246 |  0:00:02s
epoch 21 | loss: 0.26174 |  0:00:02s
epoch 22 | loss: 0.25893 |  0:00:03s
epoch 23 | loss: 0.25843 |  0:00:03s
epoch 24 | loss: 0.25495 |  0:00:03s
epoch 25 | loss: 0.25714 |  0:00:03s
epoch 26 | loss: 0.2472  |  0:00:03s
e

/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/pytorch_tabnet/abstract_model.py:82: UserWarning: Device used : cuda
  warnings.warn(f"Device used : {self.device}")
/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/pytorch_tabnet/abstract_model.py:687: UserWarning: No early stopping will be performed, last training weights will be used.
  warnings.warn(wrn_msg)


epoch 0  | loss: 0.75015 |  0:00:00s
epoch 1  | loss: 0.63318 |  0:00:00s
epoch 2  | loss: 0.61021 |  0:00:00s
epoch 3  | loss: 0.58344 |  0:00:00s
epoch 4  | loss: 0.5752  |  0:00:00s
epoch 5  | loss: 0.56885 |  0:00:00s
epoch 6  | loss: 0.54283 |  0:00:00s
epoch 7  | loss: 0.52018 |  0:00:01s
epoch 8  | loss: 0.50563 |  0:00:01s
epoch 9  | loss: 0.48631 |  0:00:01s
epoch 10 | loss: 0.4871  |  0:00:01s
epoch 11 | loss: 0.47453 |  0:00:01s
epoch 12 | loss: 0.45203 |  0:00:01s
epoch 13 | loss: 0.45521 |  0:00:01s
epoch 14 | loss: 0.45233 |  0:00:02s
epoch 15 | loss: 0.45017 |  0:00:02s
epoch 16 | loss: 0.44635 |  0:00:02s
epoch 17 | loss: 0.43702 |  0:00:02s
epoch 18 | loss: 0.43594 |  0:00:02s
epoch 19 | loss: 0.44233 |  0:00:02s
epoch 20 | loss: 0.43558 |  0:00:02s
epoch 21 | loss: 0.44107 |  0:00:02s
epoch 22 | loss: 0.42723 |  0:00:03s
epoch 23 | loss: 0.43044 |  0:00:03s
epoch 24 | loss: 0.43696 |  0:00:03s
epoch 25 | loss: 0.42875 |  0:00:03s
epoch 26 | loss: 0.42698 |  0:00:03s
e

6️⃣ FastText（Facebook 文本分类）⭐推荐

库：fasttext

优点：

专门做文本分类

非常快

安装

In [4]:
!pip install fasttext

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/88/c5/e98d9c51f3d5300d5e40ad9037dd6b3b60736fd02ab68dcc98c96be7592d/pybind11-3.0.2-py3-none-any.whl (310 kB)
  Created wheel for fasttext: filename=fasttext-0.9.3-cp38-cp38-linux_x86_64.whl size=4879147 sha256=fc59b2ead614977fdd6f460a32d47c27916cab73c0f557354d92e06db838474e
  Stored in directory: /home/wangkongqiang/.cache/pip/wheels/5f/47/ae/29d9504c5eaa6728656294cafc1ebe069a13a8c706bf1255f0
Successfully built fasttext


特点：

直接使用文本

不需要 TF-IDF

你现在的代码是 TF-IDF + LogisticRegression baseline。
如果要改成 6️⃣ FastText（Facebook 文本分类），需要做 3 个核心修改：

不再使用 TF-IDF

把训练数据转成 fastText 格式

用 fasttext.train_supervised 训练

FastText 版本完整代码

In [6]:
# ============================================
# HOPE-EXP — FASTTEXT BASELINE
# ============================================

import os
import json
import re
import zipfile
import fasttext
import numpy as np
import pandas as pd

# -------------------------
# CONFIG
# -------------------------
INPUT_DIR = "Hope-EXP datasets"

TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

OUT_DIR = os.path.join(INPUT_DIR, "baseline_fasttext_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, "submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")

FASTTEXT_TRAIN = os.path.join(OUT_DIR, "fasttext_train.txt")
FASTTEXT_TEST = os.path.join(OUT_DIR, "fasttext_test.txt")

EMOTIONS_ALL = [
    'sadness','joy','love','anger','fear','surprise',"Nuetral/unclear"
]

# -------------------------
# IO
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def build_text(df):
    return (df["title"].fillna("") + "\n\n" + df["selftext"].fillna("")).str.strip()

# -------------------------
# LOAD DATA
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)

# -------------------------
# Convert train -> FastText format
# -------------------------
def clean_label(x):
    return x.replace(" ", "_")

with open(FASTTEXT_TRAIN, "w", encoding="utf-8") as f:
    for t, lab in zip(df_train["text"], df_train["primary_label"]):
        lab = clean_label(str(lab))
        text = str(t).replace("\n", " ")
        f.write(f"__label__{lab} {text}\n")

print("FastText train file created")

df_dev["primary_label"]="Not Hope"
with open(FASTTEXT_TEST, "w", encoding="utf-8") as f:
    for t, lab in zip(df_dev["text"], df_dev["primary_label"]):
        lab = clean_label(str(lab))
        text = str(t).replace("\n", " ")
        f.write(f"__label__{lab} {text}\n")

print("FastText test file created")

# -------------------------
# Train FastText
# -------------------------
model = fasttext.train_supervised(
    input=FASTTEXT_TRAIN,
    epoch=25,
    lr=0.5,
    wordNgrams=2,
    verbose=2
)

print("FastText training finished")

# -------------------------
# Predict Task A
# -------------------------
pred_A = []

for text in df_dev["text"]:
    text = text.replace("\n"," ")
    label, prob = model.predict(text)
    lab = label[0].replace("__label__", "").replace("_"," ")
    pred_A.append(lab)

# -------------------------
# Task B (emotion simple baseline)
# -------------------------
pred_B = [["Nuetral/unclear"] for _ in range(len(df_dev))]

# -------------------------
# Simple RULE spans
# -------------------------
def extract_spans_rule(text):
    spans = []
    m = re.search(r"\bi hope\b", text.lower())
    if m:
        tail = text[m.end():].strip()
        parts = re.split(r"[.;\n]", tail)
        for p in parts:
            p = p.strip()
            if len(p) > 5:
                spans.append(p)
                break
    return spans

def rule_span_items(text):
    spans = extract_spans_rule(text)
    return [{"span":s,"outcome_stance":"Desired","actor":"Unclear"} for s in spans]

# -------------------------
# WRITE JSONL
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:

    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"],
        df_dev["lang"],
        df_dev["title"],
        df_dev["selftext"],
        df_dev["text"],
        pred_A,
        pred_B
    ):

        spans = rule_span_items(text)

        obj = {
            "row_id": int(rid),
            "lang": lang,
            "title": title,
            "selftext": selftext,
            "primary_label": plab,
            "trigger_emotions": ems,
            "span_annotations": spans
        }

        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("Prediction JSONL written")

# -------------------------
# ZIP SUBMISSION
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))

print("ZIP created:", ZIP_PATH)

FastText train file created
FastText test file created


Read 0M words
Number of words:  81684
Number of labels: 6
Progress: 100.0% words/sec/thread:  481225 lr:  0.000000 avg.loss:  0.223155 ETA:   0h 0m 0s


FastText training finished
Prediction JSONL written
ZIP created: Hope-EXP datasets/baseline_fasttext_submission/HopeEXP2026_1.zip


FastText 推荐参数（效果更好）

In [7]:
# ============================================
# HOPE-EXP — FASTTEXT BASELINE
# ============================================

import os
import json
import re
import zipfile
import fasttext
import numpy as np
import pandas as pd

# -------------------------
# CONFIG
# -------------------------
INPUT_DIR = "Hope-EXP datasets"

TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

OUT_DIR = os.path.join(INPUT_DIR, "baseline_fasttext_parameter_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, "submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")

FASTTEXT_TRAIN = os.path.join(OUT_DIR, "fasttext_train.txt")
FASTTEXT_TEST = os.path.join(OUT_DIR, "fasttext_test.txt")

EMOTIONS_ALL = [
    'sadness','joy','love','anger','fear','surprise',"Nuetral/unclear"
]

# -------------------------
# IO
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def build_text(df):
    return (df["title"].fillna("") + "\n\n" + df["selftext"].fillna("")).str.strip()

# -------------------------
# LOAD DATA
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)

# -------------------------
# Convert train -> FastText format
# -------------------------
def clean_label(x):
    return x.replace(" ", "_")

with open(FASTTEXT_TRAIN, "w", encoding="utf-8") as f:
    for t, lab in zip(df_train["text"], df_train["primary_label"]):
        lab = clean_label(str(lab))
        text = str(t).replace("\n", " ")
        f.write(f"__label__{lab} {text}\n")

print("FastText train file created")

df_dev["primary_label"]="Not Hope"
with open(FASTTEXT_TEST, "w", encoding="utf-8") as f:
    for t, lab in zip(df_dev["text"], df_dev["primary_label"]):
        lab = clean_label(str(lab))
        text = str(t).replace("\n", " ")
        f.write(f"__label__{lab} {text}\n")

print("FastText test file created")

# -------------------------
# Train FastText
# -------------------------
# model = fasttext.train_supervised(
#     input=FASTTEXT_TRAIN,
#     epoch=25,
#     lr=0.5,
#     wordNgrams=2,
#     verbose=2
# )
# 可以改成：
model = fasttext.train_supervised(
    input=FASTTEXT_TRAIN,
    epoch=50,
    lr=0.3,
    wordNgrams=3,
    dim=200,
    loss="softmax"
)
# 效果通常 比 LR 高 3-8%。

print("FastText training finished")

# -------------------------
# Predict Task A
# -------------------------
pred_A = []

for text in df_dev["text"]:
    text = text.replace("\n"," ")
    label, prob = model.predict(text)
    lab = label[0].replace("__label__", "").replace("_"," ")
    pred_A.append(lab)

# -------------------------
# Task B (emotion simple baseline)
# -------------------------
pred_B = [["Nuetral/unclear"] for _ in range(len(df_dev))]

# -------------------------
# Simple RULE spans
# -------------------------
def extract_spans_rule(text):
    spans = []
    m = re.search(r"\bi hope\b", text.lower())
    if m:
        tail = text[m.end():].strip()
        parts = re.split(r"[.;\n]", tail)
        for p in parts:
            p = p.strip()
            if len(p) > 5:
                spans.append(p)
                break
    return spans

def rule_span_items(text):
    spans = extract_spans_rule(text)
    return [{"span":s,"outcome_stance":"Desired","actor":"Unclear"} for s in spans]

# -------------------------
# WRITE JSONL
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:

    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"],
        df_dev["lang"],
        df_dev["title"],
        df_dev["selftext"],
        df_dev["text"],
        pred_A,
        pred_B
    ):

        spans = rule_span_items(text)

        obj = {
            "row_id": int(rid),
            "lang": lang,
            "title": title,
            "selftext": selftext,
            "primary_label": plab,
            "trigger_emotions": ems,
            "span_annotations": spans
        }

        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("Prediction JSONL written")

# -------------------------
# ZIP SUBMISSION
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))

print("ZIP created:", ZIP_PATH)

FastText train file created
FastText test file created


Read 0M words
Number of words:  81684
Number of labels: 6
Progress: 100.0% words/sec/thread:  125988 lr:  0.000000 avg.loss:  0.244795 ETA:   0h 0m 0s  1.4% words/sec/thread:  120999 lr:  0.295749 avg.loss:  1.747272 ETA:   0h 0m21s


FastText training finished
Prediction JSONL written
ZIP created: Hope-EXP datasets/baseline_fasttext_parameter_submission/HopeEXP2026_1.zip


7️⃣ Vowpal Wabbit

库：vowpalwabbit

优点：

超大规模数据

在线学习

安装

In [8]:
!pip install vowpalwabbit

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 5.0 MB/s eta 0:00:00a 0:00:01


1️⃣ 新增 import

在原 import 后加入：

In [11]:
from vowpalwabbit import pyvw
from vowpalwabbit import Workspace

In [14]:
# ============================================
# HOPE-EXP — SIMPLE Vowpal Wabbit BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: Workspace() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.multiclass import OneVsRestClassifier
# from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

OUT_DIR = os.path.join(INPUT_DIR, "baseline_VowpalWabbit_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
     'Hopelessness',
     'Not Hope',
     'Realistic Hope',
     'Sarcastic Hope',
     'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# vec = TfidfVectorizer()  # default params (simple baseline)
# Xtr = vec.fit_transform(df_train["text"])
# Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with Workspace default params
# -------------------------
# 2️⃣ Label mapping（VW 只能用数字标签）

# 在模型训练前加入：
# label -> id
labels_A = sorted(df_train["primary_label"].astype(str).unique())
label2id_A = {l:i+1 for i,l in enumerate(labels_A)}
id2label_A = {i:l for l,i in label2id_A.items()}

# 3️⃣ VW 输入格式函数
# VW 格式：<label> |text features
# 加入函数：
def clean_vw(text):

    text = str(text)

    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = text.replace("\t", " ")

    text = text.replace("|", " ")
    text = text.replace(":", " ")

    text = re.sub(r"\s+", " ", text)

    return text.strip()
    
def vw_format(label, text):
    # text = text.replace("|", " ")
    text = clean_vw(text)
    return f"{label} |text {text}"

# 4️⃣ Task A 训练 (primary_label)
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)

# number of classes
num_class = len(labels_A)

# vw_A = pyvw.vw(
#     f"--oaa {num_class} --loss_function logistic --passes 10 --quiet"
# )
vw_A = Workspace(
    f"--oaa {num_class} --loss_function logistic --quiet"
)
# train
for text, lab in zip(df_train["text"], df_train["primary_label"]):
    y = label2id_A[str(lab)]
    example = vw_format(y, text)
    vw_A.learn(example)

# 5️⃣ Task A 预测
# pred_A = clf_A.predict(Xdv).tolist()
pred_A = []

for text in df_dev["text"]:
    ex = vw_A.example(f"|text {text}")
    pred_id = int(vw_A.predict(ex))
    pred_A.append(id2label_A[pred_id])
    
# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + Workspace default params
# -------------------------
# mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
# Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# 6️⃣ Task B (emotion multilabel)
# VW 没有原生 multilabel，最简单方法：
# One-vs-Rest（和 sklearn 一样）

vw_B_models = {}

for emo in EMOTIONS_ALL:

    # vw = pyvw.vw("--loss_function logistic --passes 10 --quiet")
    vw = Workspace(
       f" --loss_function logistic --quiet"
    )
    for text, labs in zip(df_train["text"], df_train["trigger_emotions"]):

        y = 1 if emo in labs else -1
        ex = vw_format(y, text)
        vw.learn(ex)

    vw_B_models[emo] = vw

# 7️⃣ Emotion 预测
# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)
# pred_B = []

pred_B = []
for text in df_dev["text"]:

    labs = []

    for emo, vw in vw_B_models.items():

        ex = vw.example(f"|text {text}")
        score = vw.predict(ex)

        if score > 0:
            labs.append(emo)

    if not labs:
        labs = ["Nuetral/unclear"]

    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]

    pred_B.append(labs)
    
# for row in pred_B_bin:
#     labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
#     if not labs:
#         labs = ["Nuetral/unclear"]
#     if "Nuetral/unclear" in labs and len(labs) > 1:
#         labs = [z for z in labs if z != "Nuetral/unclear"]
#     pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory


✅ Wrote prediction JSONL!


再给你一个 HOPE-EXP VW 强化版参数（推荐）

In [17]:
# ============================================
# HOPE-EXP — SIMPLE Vowpal Wabbit BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: Workspace() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

# from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
# from sklearn.multiclass import OneVsRestClassifier
# from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

OUT_DIR = os.path.join(INPUT_DIR, "baseline_VowpalWabbit_parameter_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
     'Hopelessness',
     'Not Hope',
     'Realistic Hope',
     'Sarcastic Hope',
     'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# vec = TfidfVectorizer()  # default params (simple baseline)
# Xtr = vec.fit_transform(df_train["text"])
# Xdv = vec.transform(df_dev["text"])

# -------------------------
# Task A: primary_label (multiclass) with Workspace default params
# -------------------------
# 2️⃣ Label mapping（VW 只能用数字标签）

# 在模型训练前加入：
# label -> id
labels_A = sorted(df_train["primary_label"].astype(str).unique())
label2id_A = {l:i+1 for i,l in enumerate(labels_A)}
id2label_A = {i:l for l,i in label2id_A.items()}

# 3️⃣ VW 输入格式函数
# VW 格式：<label> |text features
# 加入函数：
def clean_vw(text):

    text = str(text)

    text = text.replace("\n", " ")
    text = text.replace("\r", " ")
    text = text.replace("\t", " ")

    text = text.replace("|", " ")
    text = text.replace(":", " ")

    text = re.sub(r"\s+", " ", text)

    return text.strip()
    
def vw_format(label, text):
    # text = text.replace("|", " ")
    text = clean_vw(text)
    return f"{label} |text {text}"

# 4️⃣ Task A 训练 (primary_label)
# clf_A = LogisticRegression()  # default params
# clf_A.fit(Xtr, df_train["primary_label"].astype(str).values)

# number of classes
num_class = len(labels_A)

# vw_A = pyvw.vw(
#     f"--oaa {num_class} --loss_function logistic --passes 10 --quiet"
# )
# vw_A = Workspace(
#     f"--oaa {num_class} --loss_function logistic --quiet"
# )

vw_A = Workspace(
    f"""
    --oaa {num_class}
    --loss_function logistic
    --ngram 2
    --skips 1
    -b 26
    -l 0.5
    --adaptive
    --invariant
    --quiet
    """
)
# train
for text, lab in zip(df_train["text"], df_train["primary_label"]):
    y = label2id_A[str(lab)]
    example = vw_format(y, text)
    vw_A.learn(example)

# 5️⃣ Task A 预测

# pred_A = clf_A.predict(Xdv).tolist()
pred_A = []

for text in df_dev["text"]:
    ex = vw_A.example(f"|text {text}")
    pred_id = int(vw_A.predict(ex))
    pred_A.append(id2label_A[pred_id])
    
# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + Workspace default params
# -------------------------
# mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
# Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# 6️⃣ Task B (emotion multilabel)
# VW 没有原生 multilabel，最简单方法：
# One-vs-Rest（和 sklearn 一样）

vw_B_models = {}

for emo in EMOTIONS_ALL:

    # vw = pyvw.vw("--loss_function logistic --passes 10 --quiet")
    # vw = Workspace(
    #    f" --loss_function logistic --quiet"
    # )
    vw = Workspace(
       f" --loss_function logistic --ngram 2 --skips 1 -b 26 -l 0.5 --adaptive --invariant --quiet"
    )
    for text, labs in zip(df_train["text"], df_train["trigger_emotions"]):

        y = 1 if emo in labs else -1
        ex = vw_format(y, text)
        vw.learn(ex)

    vw_B_models[emo] = vw

# 7️⃣ Emotion 预测
# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B.fit(Xtr, Ytr)
# pred_B_bin = clf_B.predict(Xdv)
# pred_B = []

pred_B = []

for text in df_dev["text"]:

    labs = []

    for emo, vw in vw_B_models.items():

        ex = vw.example(f"|text {text}")
        score = vw.predict(ex)

        if score > 0:
            labs.append(emo)

    if not labs:
        labs = ["Nuetral/unclear"]

    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]

    pred_B.append(labs)
    
# for row in pred_B_bin:
#     labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
#     if not labs:
#         labs = ["Nuetral/unclear"]
#     if "Nuetral/unclear" in labs and len(labs) > 1:
#         labs = [z for z in labs if z != "Nuetral/unclear"]
#     pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory


✅ Wrote prediction JSONL!


通常 比 LogisticRegression baseline 提升 3%+。

8️⃣ ThunderSVM

库：thundersvm

特点：

GPU 加速 SVM

sklearn API 兼容

安装

In [8]:
!pip install thundersvm-cpu

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple


import 导入

In [2]:
from thundersvm import SVC
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import LabelEncoder

先确认你的 CUDA 版本

In [3]:
!nvidia-smi

Sun Mar 22 21:07:43 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 575.55.01              Driver Version: 576.40         CUDA Version: 12.9     |
|-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  NVIDIA GeForce RTX 3090        On  |   00000000:01:00.0 Off |                  N/A |
|  0%   49C    P8             34W /  390W |    1533MiB /  24576MiB |     36%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [6]:
# ============================================
# HOPE-EXP — SIMPLE ThunderSVM BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: SVC() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
# from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

OUT_DIR = os.path.join(INPUT_DIR, "baseline_ThunderSVM_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
     'Hopelessness',
     'Not Hope',
     'Realistic Hope',
     'Sarcastic Hope',
     'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
vec = TfidfVectorizer(    
    max_features=2000,   # 限制特征数量（最重要优化）
    min_df=2,             # 出现至少2次
    max_df=0.9,           # 过滤高频词
    ngram_range=(1,1)     # 只用 unigram
)  # default params (simple baseline)
Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

svd = TruncatedSVD(n_components=300)
Xtr_reduced = svd.fit_transform(Xtr)
Xdv_reduced = svd.transform(Xdv)
# -------------------------
# Task A: primary_label (multiclass) with SVC default params
# -------------------------
# clf_A = LogisticRegression()  # default params
le = LabelEncoder()
y_train = le.fit_transform(df_train["primary_label"].astype(str))
clf_A = SVC(kernel='rbf')  # default params
# clf_A = SVC(kernel='linear')  # default params
clf_A.fit(Xtr_reduced, y_train)
pred_A = clf_A.predict(Xdv_reduced).tolist()
# 如果需要恢复成原始标签
pred_A = np.array(pred_A).astype(int)
pred_A = le.inverse_transform(pred_A)

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + SVC default params
# -------------------------
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
clf_B = OneVsRestClassifier(SVC(kernel='rbf'))  # default params
# clf_B = OneVsRestClassifier(SVC(kernel='linear'))  # default params
clf_B.fit(Xtr_reduced, Ytr)
pred_B_bin = clf_B.predict(Xdv_reduced)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


In [3]:
# ============================================
# HOPE-EXP — SIMPLE ThunderSVM BASELINE (SUBMISSION)
# - Train: HopeEXP_train.jsonl
# - Predict: HopeEXP_test_unlabeled.jsonl
# - Model: SVC() default params
# - Output: TeamName_RunNumber.zip (no subdir) containing a .jsonl
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

# from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import HashingVectorizer
# from sklearn.linear_model import LogisticRegression
from sklearn.multiclass import OneVsRestClassifier
from sklearn.preprocessing import MultiLabelBinarizer

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

OUT_DIR = os.path.join(INPUT_DIR, "baseline_ThunderSVM_HashingVectorizer_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    # training only (gold). dev_unlabeled has none.
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    # if neutral + others => drop neutral
    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    # dedup preserve order
    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# Simple RULE spans/stance/actor (fast)
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()

    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope',
     'Hopelessness',
     'Not Hope',
     'Realistic Hope',
     'Sarcastic Hope',
     'Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# Load TRAIN (gold) + DEV (unlabeled)
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)  # internal only (not saved)

# training labels
if "primary_label" not in df_train.columns:
    raise ValueError("Train gold is missing primary_label.")
if "trigger_emotions" not in df_train.columns:
    raise ValueError("Train gold is missing trigger_emotions.")
df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# Features: TF-IDF (default)
# -------------------------
# vec = TfidfVectorizer(    
#     max_features=2000,   # 限制特征数量（最重要优化）
#     min_df=2,             # 出现至少2次
#     max_df=0.9,           # 过滤高频词
#     ngram_range=(1,1)     # 只用 unigram
# )  # default params (simple baseline)
# Xtr = vec.fit_transform(df_train["text"])
# Xdv = vec.transform(df_dev["text"])

vec = HashingVectorizer(
    n_features=2**12,      # ≈ 4096 维（可调：2**14 更强）
    alternate_sign=False,  # 避免符号冲突（推荐设为 False）
    ngram_range=(1,1),
    norm='l2'
)

Xtr = vec.transform(df_train["text"])  # ⚠️ 没有 fit！
Xdv = vec.transform(df_dev["text"])

svd = TruncatedSVD(n_components=300)
Xtr_reduced = svd.fit_transform(Xtr)
Xdv_reduced = svd.transform(Xdv)
# -------------------------
# Task A: primary_label (multiclass) with SVC default params
# -------------------------
# clf_A = LogisticRegression()  # default params
le = LabelEncoder()
y_train = le.fit_transform(df_train["primary_label"].astype(str))
# clf_A = SVC(kernel='rbf')  # default params
clf_A = SVC(kernel='linear')  # default params
clf_A.fit(Xtr_reduced, y_train)
pred_A = clf_A.predict(Xdv_reduced).tolist()
# 如果需要恢复成原始标签
pred_A = np.array(pred_A).astype(int)
pred_A = le.inverse_transform(pred_A)

# -------------------------
# Task B: trigger_emotions (multi-label) with OVR + SVC default params
# -------------------------
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

# clf_B = OneVsRestClassifier(LogisticRegression())  # default params
# clf_B = OneVsRestClassifier(SVC(kernel='rbf'))  # default params
clf_B = OneVsRestClassifier(SVC(kernel='linear'))  # default params
clf_B.fit(Xtr_reduced, Ytr)
pred_B_bin = clf_B.predict(Xdv_reduced)

pred_B = []
for row in pred_B_bin:
    labs = [mlb.classes_[i] for i, v in enumerate(row) if v == 1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# Write ONE prediction JSONL (submission file content)
# Schema matches gold (no "text")
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),  # internal only
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# Zip for submission: TeamName_RunNumber.zip (no subdir)
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))  # no subdirectory

✅ Wrote prediction JSONL!


9️⃣ H2O AutoML

库：h2o

特点：

自动训练多个模型

自动调参

安装

In [1]:
pip install -f http://h2o-release.s3.amazonaws.com/h2o/latest_stable_Py.html h2o

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
Looking in links: http://h2o-release.s3.amazonaws.com/h2o/latest_stable_Py.html
Note: you may need to restart the kernel to use updated packages.


支持模型：

GBM

RandomForest

DeepLearning

StackedEnsemble

导入库（新增）

在原代码 import 部分加入：

In [2]:
# ============================================
# HOPE-EXP — COMPLETE H2O AutoML BASELINE (SUBMISSION)
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.decomposition import TruncatedSVD

import h2o
from h2o.automl import H2OAutoML

# -------------------------
# CONFIG (EDIT THESE)
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

OUT_DIR = os.path.join(INPUT_DIR, "baseline_H2OAutoML_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, f"submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")  # TeamName_RunNumber.zip

EMOTIONS_ALL = ['sadness', 'joy', 'love', 'anger', 'fear', 'surprise', "Nuetral/unclear"]

# -------------------------
# IO helpers
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path, "r", encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except Exception:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("").astype(str) + "\n\n" + df["selftext"].fillna("").astype(str)).str.strip()

def normalize_emotions_list(x):
    if isinstance(x, list):
        labs = [str(z).strip() for z in x if str(z).strip()]
    elif x is None or (isinstance(x, float) and np.isnan(x)):
        labs = []
    else:
        labs = [str(x).strip()]

    all_low = {a.lower(): a for a in EMOTIONS_ALL}
    out = []
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(), "Nuetral/unclear"))

    if not out:
        out = ["Nuetral/unclear"]

    if "Nuetral/unclear" in out and len(set(out)) > 1:
        out = [z for z in out if z != "Nuetral/unclear"]

    seen = set()
    ded = []
    for z in out:
        if z not in seen:
            ded.append(z)
            seen.add(z)
    return ded

# -------------------------
# RULE-BASED spans/stance/actor
# -------------------------
HOPE_CUES = [
    r"\bi hope\b", r"\bhope\b", r"\bhoping\b", r"\bhopefully\b",
    r"\bi wish\b", r"\bwish\b",
    r"\bespero que\b", r"\bojal[aá]\b",
    r"\bpls\b", r"\bplease\b", r"\bpraying\b"
]
NEG_CUES = {"not","don't","dont","doesn't","doesnt","never","no","won't","wont","reject","ruin","die"}
WORLD_KWS = {"rain","weather","economy","inflation","rent","surgery","meds","medicine","treatment",
             "appointment","doctor","clinic","therapy","visa","approval","approved","respond","reply","system"}
OTHER_KWS = {"they","them","landlord","boss","manager","school","university","company",
             "immigration","gov","government","committee","office"}

def extract_spans_rule(text: str, max_spans: int = 3):
    t = (text or "").strip()
    if not t:
        return []
    low = t.lower()
    cue_end = None
    for pat in HOPE_CUES:
        m = re.search(pat, low)
        if m:
            cue_end = m.end()
            break
    if cue_end is None:
        return []

    tail = t[cue_end:].strip(" :,-—. \n\t")
    if not tail:
        return []

    parts = re.split(r"\b(?:and|so|that|but)\b|[.;\n]", tail, flags=re.IGNORECASE)
    parts = [p.strip(" .,:;!?\"'()[]{}") for p in parts]
    parts = [p for p in parts if len(p) >= 3]

    spans = []
    for p in parts:
        if len(spans) >= max_spans:
            break
        if p and p in t:
            spans.append(p)

    if not spans and parts:
        fb = parts[0]
        if fb and fb in t:
            spans = [fb]

    seen = set()
    out = []
    for s in spans:
        if s not in seen:
            out.append(s)
            seen.add(s)
    return out[:max_spans]

def stance_rule(span: str) -> str:
    s = (span or "").lower()
    for c in NEG_CUES:
        if re.search(rf"\b{re.escape(c)}\b", s):
            return "Avoided"
    return "Desired"

def actor_rule(span: str) -> str:
    s = (span or "").lower().strip()
    if re.match(r"^(it|this|that)\b", s):
        return "Unclear"
    if re.search(r"\b(i|i'm|im|i’ll|ill|i will|my|me)\b", s):
        return "Self"
    if any(k in s for k in WORLD_KWS):
        return "World/System"
    for k in OTHER_KWS:
        if re.search(rf"\b{re.escape(k)}\b", s):
            return "Other"
    return "Unclear"

def rule_span_items(text: str, primary_label: str):
    if primary_label in ['General Hope','Hopelessness','Not Hope','Realistic Hope','Sarcastic Hope','Unrealistic Hope']:
        return []
    spans = extract_spans_rule(text)
    return [{"span": sp, "outcome_stance": stance_rule(sp), "actor": actor_rule(sp)} for sp in spans]

# -------------------------
# LOAD DATA
# -------------------------
df_train = load_jsonl(TRAIN_GOLD_JSONL)
df_dev   = load_jsonl(DEV_UNLAB_JSONL)

for _df in (df_train, df_dev):
    if "row_id" not in _df.columns:
        raise ValueError("Missing row_id in input JSONL.")
    if "lang" not in _df.columns:
        _df["lang"] = None
    _df["title"] = _df["title"].fillna("")
    _df["selftext"] = _df["selftext"].fillna("")
    _df["text"] = build_text(_df)

df_train["trigger_emotions"] = df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# FEATURES: TF-IDF + SVD
# -------------------------
vec = TfidfVectorizer(max_features=2000, min_df=2, max_df=0.9, ngram_range=(1,1))
Xtr = vec.fit_transform(df_train["text"])
Xdv = vec.transform(df_dev["text"])

svd = TruncatedSVD(n_components=300)
Xtr_reduced = svd.fit_transform(Xtr)
Xdv_reduced = svd.transform(Xdv)

train_df = pd.DataFrame(Xtr_reduced)
dev_df = pd.DataFrame(Xdv_reduced)

# -------------------------
# H2O INIT
# -------------------------
h2o.init(    
    ip="127.0.0.1",
    port=54321,
    max_mem_size="4G",
    nthreads=-1
)

# -------------------------
# Task A: primary_label (multi-class)
# -------------------------
train_df_A = train_df.copy()
train_df_A["label"] = df_train["primary_label"].astype(str).values

h2o_train_A = h2o.H2OFrame(train_df_A)
h2o_dev_A   = h2o.H2OFrame(dev_df)
h2o_train_A["label"] = h2o_train_A["label"].asfactor()

x_cols = list(train_df.columns)
y_col = "label"

aml_A = H2OAutoML(max_models=1, seed=42, exclude_algos=["DeepLearning"], verbosity="info")
aml_A.train(x=x_cols, y=y_col, training_frame=h2o_train_A)

pred_A = aml_A.leader.predict(h2o_dev_A).as_data_frame()["predict"].tolist()

# -------------------------
# Task B: trigger_emotions (multi-label)
# -------------------------
mlb = MultiLabelBinarizer(classes=EMOTIONS_ALL)
Ytr = mlb.fit_transform(df_train["trigger_emotions"])

pred_B = []
emotion_models = {}
h2o_dev_B = h2o.H2OFrame(dev_df)

for i, emotion in enumerate(EMOTIONS_ALL):
    train_df_B = train_df.copy()
    train_df_B["label"] = Ytr[:, i]

    h2o_train_B = h2o.H2OFrame(train_df_B)
    h2o_train_B["label"] = h2o_train_B["label"].asfactor()

    aml_B = H2OAutoML(max_models=1, seed=42, exclude_algos=["DeepLearning"])
    aml_B.train(x=x_cols, y="label", training_frame=h2o_train_B)

    emotion_models[emotion] = aml_B.leader

pred_matrix = []
for emotion in EMOTIONS_ALL:
    model = emotion_models[emotion]
    pred = model.predict(h2o_dev_B)["predict"].as_data_frame().values.flatten()
    pred_matrix.append(pred)

pred_matrix = np.array(pred_matrix).T

for row in pred_matrix:
    labs = [EMOTIONS_ALL[i] for i,v in enumerate(row) if int(v)==1]
    if not labs:
        labs = ["Nuetral/unclear"]
    if "Nuetral/unclear" in labs and len(labs) > 1:
        labs = [z for z in labs if z != "Nuetral/unclear"]
    pred_B.append(labs)

# -------------------------
# WRITE JSONL SUBMISSION
# -------------------------
with open(PRED_JSONL, "w", encoding="utf-8") as f:
    for rid, lang, title, selftext, text, plab, ems in zip(
        df_dev["row_id"].astype(int).tolist(),
        df_dev["lang"].tolist(),
        df_dev["title"].tolist(),
        df_dev["selftext"].tolist(),
        df_dev["text"].tolist(),
        pred_A,
        pred_B
    ):
        spans = rule_span_items(text, str(plab))
        obj = {
            "row_id": int(rid),
            "lang": json_safe_lang(lang),
            "title": title,
            "selftext": selftext,
            "primary_label": str(plab),
            "trigger_emotions": ems,
            "span_annotations": spans
        }
        f.write(json.dumps(obj, ensure_ascii=False) + "\n")

print("✅ Wrote prediction JSONL!")

# -------------------------
# ZIP FOR SUBMISSION
# -------------------------
with zipfile.ZipFile(ZIP_PATH, "w", compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL, arcname=os.path.basename(PRED_JSONL))

print(f"✅ Created submission ZIP: {ZIP_PATH}")

# -------------------------
# CLEAN H2O
# -------------------------
h2o.remove_all()
h2o.shutdown(prompt=False)

Checking whether there is an H2O instance running at http://127.0.0.1:54321..... not found.
Attempting to start a local H2O server...
; Java HotSpot(TM) 64-Bit Server VM (build 26+35-2893, mixed mode, sharing)
  Starting server from C:\Users\8888\Anaconda3\envs\H2O\lib\site-packages\h2o\backend\bin\h2o.jar
  Ice root: C:\Users\8888\AppData\Local\Temp\tmp4oimzadv
  JVM stdout: C:\Users\8888\AppData\Local\Temp\tmp4oimzadv\h2o_8888_started_from_python.out
  JVM stderr: C:\Users\8888\AppData\Local\Temp\tmp4oimzadv\h2o_8888_started_from_python.err
  Server is running at http://127.0.0.1:54321
Connecting to H2O server at http://127.0.0.1:54321 ... successful.
Please download and install the latest version from: https://h2o-release.s3.amazonaws.com/h2o/latest_stable.html


H2O_cluster_uptime:,01 secs
H2O_cluster_timezone:,Asia/Shanghai
H2O_data_parsing_timezone:,UTC
H2O_cluster_version:,3.46.0.9
H2O_cluster_version_age:,4 months and 1 day
H2O_cluster_name:,H2O_from_python_8888_9kubqz
H2O_cluster_total_nodes:,1
H2O_cluster_free_memory:,3.983 Gb
H2O_cluster_total_cores:,20
H2O_cluster_allowed_cores:,20
H2O_cluster_status:,"locked, healthy"


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
14:50:49.555: Project: AutoML_1_20260326_145049
14:50:49.559: 5-fold cross-validation will be used.
14:50:49.562: Setting stopping tolerance adaptively based on the training frame: 0.014348812093082914
14:50:49.562: Build control seed: 42
14:50:49.564: training frame: Frame key: AutoML_1_20260326_145049_training_py_1_sid_b1b7    cols: 301    rows: 4857  chunks: 8    size: 11829129  checksum: -5498367417992617109
14:50:49.564: validation frame: NULL
14:50:49.564: leaderboard frame: NULL
14:50:49.564: blending frame: NULL
14:50:49.564: response column: label
14:50:49.564: fold column: null
14:50:49.564: weights column: null
14:50:49.579: AutoML: XGBoost is not available; skipping it.
14:50:49.586: Loading execution steps: [{XGBoost : [def_2 (1g, 10w), def_1 (2g, 10w), def_3 (3g, 10w

C:\Users\8888\Anaconda3\envs\H2O\lib\site-packages\h2o\frame.py:1985: H2ODependencyWarning: Converting H2O frame to pandas dataframe using single-thread.  For faster conversion using multi-thread, install polars and pyarrow and use it as pandas_df = h2o_df.as_data_frame(use_multi_thread=True)

  "pandas_df = h2o_df.as_data_frame(use_multi_thread=True)\n", H2ODependencyWarning)


Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
14:52:38.580: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
14:52:50.279: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
14:53:00.185: AutoML: XGBoost is not available; skipping it.

███████████████████████████████████████████████████████████████| (done) 100%
Parse progress: |████████████████████████████████████████████████████████████████| (done) 100%
AutoML progress: |
14:53:10.209: AutoML: XGBoost is

C:\Users\8888\Anaconda3\envs\H2O\lib\site-packages\ipykernel_launcher.py:306: H2ODeprecationWarning: Deprecated, use ``h2o.cluster().shutdown()``.


In [3]:
!java -version

java version "26" 2026-03-17
Java(TM) SE Runtime Environment (build 26+35-2893)
Java HotSpot(TM) 64-Bit Server VM (build 26+35-2893, mixed mode, sharing)


In [2]:
import h2o
print(h2o.connection())

None


查看 WSL2 的 IP

在 WSL 终端运行：

In [3]:
!ip addr

1: lo: <LOOPBACK,UP,LOWER_UP> mtu 65536 qdisc noqueue state UNKNOWN group default qlen 1000
    link/loopback 00:00:00:00:00:00 brd 00:00:00:00:00:00
    inet 127.0.0.1/8 scope host lo
       valid_lft forever preferred_lft forever
    inet 10.255.255.254/32 brd 10.255.255.254 scope global lo
       valid_lft forever preferred_lft forever
    inet6 ::1/128 scope host 
       valid_lft forever preferred_lft forever
2: eth0: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 1500 qdisc mq state UP group default qlen 1000
    link/ether 00:15:5d:8a:c7:0e brd ff:ff:ff:ff:ff:ff
    inet 172.18.49.131/20 brd 172.18.63.255 scope global eth0
       valid_lft forever preferred_lft forever
    inet6 fe80::215:5dff:fe8a:c70e/64 scope link 
       valid_lft forever preferred_lft forever


或者更简单：

In [4]:
!hostname -I

172.18.49.131 


这个就是 WSL2 的 IP。

然后在 Windows 浏览器打开：

http://172.18.49.131:54321

建议先测试 H2O 是否正常

In [ ]:
import h2o
h2o.init()
h2o.cluster_status()

Checking whether there is an H2O instance running at http://localhost:54321. connected.


安装

In [3]:
!pip install autogluon.tabular==1.1.1

Looking in indexes: https://pypi.tuna.tsinghua.edu.cn/simple
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/7a/25/0e022e7a1dce34bd6b9e015d19e3ac317869439864a5e34995a65050c720/autogluon.tabular-1.1.1-py3-none-any.whl (312 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/5c/bb/cd3d9dbb736acf75bf711ee76401a95339807bf9c478eff7b977bd23ecc6/autogluon.core-1.1.1-py3-none-any.whl (234 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/c9/18/c6749ba777564d735218023c0a38a80d03ba2ec61165b5144eb35d6c0588/autogluon.features-1.1.1-py3-none-any.whl (63 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/d3/87/8189f22ee798177bc7b40afd13f046442c5f91b699e70a950b42ff447e80/boto3-1.37.38-py3-none-any.whl (139 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/43/8f/c1844dd06604c085ada16dcfbd7ad71fc7f9b9ea15365aeacf4da5c9d8e3/autogluon.common-1.1.1-py3-none-any.whl (64 kB)
  Using cached https://pypi.tuna.tsinghua.edu.cn/packages/55/1b/93f3504afc7c523

✅ HOPE-EXP — AutoGluon 强力版 Baseline

In [4]:
# ============================================
# HOPE-EXP — AutoGluon STRONG BASELINE
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.decomposition import TruncatedSVD

from autogluon.tabular import TabularPredictor

# -------------------------
# CONFIG
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

OUT_DIR = os.path.join(INPUT_DIR, "baseline_AutoGluon_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, "submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")

EMOTIONS_ALL = ['sadness','joy','love','anger','fear','surprise',"Nuetral/unclear"]

# -------------------------
# IO
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("")+"\n\n"+df["selftext"].fillna("")).str.strip()

def normalize_emotions_list(x):

    if isinstance(x,list):
        labs=[str(z).strip() for z in x if str(z).strip()]
    else:
        labs=[str(x).strip()] if x else []

    all_low={a.lower():a for a in EMOTIONS_ALL}

    out=[]
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(),"Nuetral/unclear"))

    if not out:
        out=["Nuetral/unclear"]

    if "Nuetral/unclear" in out and len(out)>1:
        out=[z for z in out if z!="Nuetral/unclear"]

    return list(dict.fromkeys(out))

# -------------------------
# RULE spans
# -------------------------

HOPE_CUES=[r"\bi hope\b",r"\bhope\b",r"\bwish\b",r"\bhopefully\b"]

NEG_CUES={"not","never","no"}

WORLD_KWS={"weather","economy","doctor","clinic","therapy"}

OTHER_KWS={"they","landlord","boss","manager","company"}

def extract_spans_rule(text,max_spans=3):

    t=text.strip()
    low=t.lower()

    cue_end=None
    for pat in HOPE_CUES:
        m=re.search(pat,low)
        if m:
            cue_end=m.end()
            break

    if cue_end is None:
        return []

    tail=t[cue_end:]

    parts=re.split(r"\b(?:and|but|so|that)\b|[.;\n]",tail)

    parts=[p.strip(" .,!?") for p in parts if len(p)>2]

    return parts[:max_spans]

def stance_rule(span):

    s=span.lower()

    for c in NEG_CUES:
        if c in s:
            return "Avoided"

    return "Desired"

def actor_rule(span):

    s=span.lower()

    if re.search(r"\b(i|my|me)\b",s):
        return "Self"

    if any(k in s for k in WORLD_KWS):
        return "World/System"

    if any(k in s for k in OTHER_KWS):
        return "Other"

    return "Unclear"

def rule_span_items(text,label):

    if label in ['General Hope','Hopelessness','Not Hope',
                 'Realistic Hope','Sarcastic Hope','Unrealistic Hope']:
        return []

    spans=extract_spans_rule(text)

    return [
        {"span":sp,
         "outcome_stance":stance_rule(sp),
         "actor":actor_rule(sp)}
        for sp in spans
    ]

# -------------------------
# LOAD DATA
# -------------------------

df_train=load_jsonl(TRAIN_GOLD_JSONL)
df_dev=load_jsonl(DEV_UNLAB_JSONL)

for df in (df_train,df_dev):

    if "lang" not in df.columns:
        df["lang"]=None

    df["title"]=df["title"].fillna("")
    df["selftext"]=df["selftext"].fillna("")
    df["text"]=build_text(df)

df_train["trigger_emotions"]=df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# TFIDF + SVD
# -------------------------

vec=TfidfVectorizer(max_features=2000,min_df=2,max_df=0.9)

Xtr=vec.fit_transform(df_train["text"])
Xdv=vec.transform(df_dev["text"])

svd=TruncatedSVD(n_components=100)

Xtr_reduced=svd.fit_transform(Xtr)
Xdv_reduced=svd.transform(Xdv)

train_df=pd.DataFrame(Xtr_reduced)
dev_df=pd.DataFrame(Xdv_reduced)

# =====================================================
# Task A — primary_label
# =====================================================

train_A=train_df.copy()
train_A["label"]=df_train["primary_label"].astype(str)

predictor_A=TabularPredictor(
    label="label",
    path=os.path.join(OUT_DIR,"model_A")
).fit(
    train_A,
    presets="best_quality"
)

pred_A=predictor_A.predict(dev_df).tolist()

# =====================================================
# Task B — multi-label emotions
# =====================================================

mlb=MultiLabelBinarizer(classes=EMOTIONS_ALL)

Ytr=mlb.fit_transform(df_train["trigger_emotions"])

emotion_models={}

pred_B=[]

for i,emotion in enumerate(EMOTIONS_ALL):

    train_B=train_df.copy()
    train_B["label"]=Ytr[:,i]

    predictor=TabularPredictor(
        label="label",
        path=os.path.join(OUT_DIR,f"emotion_{emotion}")
    ).fit(
        train_B,
        presets="best_quality"
    )

    emotion_models[emotion]=predictor

pred_matrix=[]

for emotion in EMOTIONS_ALL:

    model=emotion_models[emotion]

    pred=model.predict(dev_df).values

    pred_matrix.append(pred)

pred_matrix=np.array(pred_matrix).T

for row in pred_matrix:

    labs=[EMOTIONS_ALL[i] for i,v in enumerate(row) if int(v)==1]

    if not labs:
        labs=["Nuetral/unclear"]

    if "Nuetral/unclear" in labs and len(labs)>1:
        labs=[z for z in labs if z!="Nuetral/unclear"]

    pred_B.append(labs)

# -------------------------
# WRITE JSONL
# -------------------------

with open(PRED_JSONL,"w",encoding="utf-8") as f:

    for rid,lang,title,selftext,text,plab,ems in zip(
        df_dev["row_id"],
        df_dev["lang"],
        df_dev["title"],
        df_dev["selftext"],
        df_dev["text"],
        pred_A,
        pred_B
    ):

        spans=rule_span_items(text,str(plab))

        obj={
            "row_id":int(rid),
            "lang":json_safe_lang(lang),
            "title":title,
            "selftext":selftext,
            "primary_label":str(plab),
            "trigger_emotions":ems,
            "span_annotations":spans
        }

        f.write(json.dumps(obj,ensure_ascii=False)+"\n")

print("✅ JSONL created")

# -------------------------
# ZIP
# -------------------------

with zipfile.ZipFile(ZIP_PATH,"w",compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL,arcname=os.path.basename(PRED_JSONL))

print("✅ ZIP created:",ZIP_PATH)

/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.1.1
Python Version:     3.8.20
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          20
Memory Avail:       21.06 GB / 23.38 GB (90.1%)
Disk Space Avail:   486.65 GB / 1862.13 GB (26.1%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determ

[1000]	valid_set's multi_error: 0.224074


	0.7811	 = Validation score   (accuracy)
	41.64s	 = Training   runtime
	0.09s	 = Validation runtime
Fitting model: RandomForestGini_BAG_L1 ... Training model for up to 517.03s of the 817.07s of remaining time.
	0.7264	 = Validation score   (accuracy)
	0.9s	 = Training   runtime
	0.13s	 = Validation runtime
Fitting model: RandomForestEntr_BAG_L1 ... Training model for up to 515.68s of the 815.72s of remaining time.
	0.729	 = Validation score   (accuracy)
	1.05s	 = Training   runtime
	0.13s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ... Training model for up to 514.2s of the 814.24s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy
	Ran out of time, early stopping on iteration 1136.
	Ran out of time, early stopping on iteration 1154.
	Ran out of time, early stopping on iteration 1200.
	Ran out of time, early stopping on iteration 1336.
	Ran out of time, early stopping on iteration 1646.
	0.7765	 = Validation score   (accu

[1000]	valid_set's multi_error: 0.242175


	0.777	 = Validation score   (accuracy)
	37.58s	 = Training   runtime
	0.1s	 = Validation runtime
Fitting model: RandomForestGini_BAG_L1 ... Training model for up to 1721.32s of the 2619.01s of remaining time.
	0.7348	 = Validation score   (accuracy)
	0.88s	 = Training   runtime
	0.15s	 = Validation runtime
Fitting model: RandomForestEntr_BAG_L1 ... Training model for up to 1719.98s of the 2617.68s of remaining time.
	0.7274	 = Validation score   (accuracy)
	1.13s	 = Training   runtime
	0.16s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ... Training model for up to 1718.38s of the 2616.08s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy
	0.7789	 = Validation score   (accuracy)
	407.1s	 = Training   runtime
	0.02s	 = Validation runtime
Fitting model: ExtraTreesGini_BAG_L1 ... Training model for up to 1311.02s of the 2208.72s of remaining time.
	0.7307	 = Validation score   (accuracy)
	0.51s	 = Training   runtime
	0.15s	 

...

[1000]	valid_set's binary_error: 0.133443
[1000]	valid_set's binary_error: 0.159802
[1000]	valid_set's binary_error: 0.14827
[1000]	valid_set's binary_error: 0.140033


	0.8524	 = Validation score   (accuracy)
	23.82s	 = Training   runtime
	0.05s	 = Validation runtime
Fitting model: RandomForest_r39_BAG_L1 ... Training model for up to 1068.07s of the 1967.68s of remaining time.
	0.8464	 = Validation score   (accuracy)
	6.93s	 = Training   runtime
	0.45s	 = Validation runtime
Fitting model: CatBoost_r167_BAG_L1 ... Training model for up to 1059.2s of the 1958.81s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy
	0.8551	 = Validation score   (accuracy)
	28.17s	 = Training   runtime
	0.02s	 = Validation runtime
Fitting model: NeuralNetFastAI_r95_BAG_L1 ... Training model for up to 1030.29s of the 1929.91s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.1.1`. 
Fitting model: NeuralNetTorch_r41_BAG_L1 ... Training model for up to 1030.13s of the 1

[1000]	valid_set's binary_error: 0.157895


	0.8526	 = Validation score   (accuracy)
	33.89s	 = Training   runtime
	0.02s	 = Validation runtime
Fitting model: RandomForest_r127_BAG_L1 ... Training model for up to 720.42s of the 1620.04s of remaining time.
	0.8485	 = Validation score   (accuracy)
	10.78s	 = Training   runtime
	0.12s	 = Validation runtime
Fitting model: NeuralNetFastAI_r134_BAG_L1 ... Training model for up to 709.43s of the 1609.04s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy
		Import fastai failed. A quick tip is to install via `pip install autogluon.tabular[fastai]==1.1.1`. 
Fitting model: RandomForest_r34_BAG_L1 ... Training model for up to 709.27s of the 1608.89s of remaining time.
	0.8411	 = Validation score   (accuracy)
	2.92s	 = Training   runtime
	0.11s	 = Validation runtime
Fitting model: LightGBM_r94_BAG_L1 ... Training model for up to 706.17s of the 1605.79s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLo

✅ JSONL created
✅ ZIP created: Hope-EXP datasets/baseline_AutoGluon_submission/HopeEXP2026_1.zip


In [4]:
# ============================================
# HOPE-EXP — AutoGluon STRONG BASELINE
# ============================================

import os, json, re, zipfile
import numpy as np
import pandas as pd

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.preprocessing import MultiLabelBinarizer
from sklearn.decomposition import TruncatedSVD

from autogluon.tabular import TabularPredictor

# -------------------------
# CONFIG
# -------------------------
INPUT_DIR = "Hope-EXP datasets"
TRAIN_GOLD_JSONL = os.path.join(INPUT_DIR, "HopeEXP_Train.jsonl")
DEV_UNLAB_JSONL  = os.path.join(INPUT_DIR, "HopeEXP_Test_unlabeled.jsonl")

OUT_DIR = os.path.join(INPUT_DIR, "baseline_AutoGluon_TruncatedSVD_submission")
os.makedirs(OUT_DIR, exist_ok=True)

TEAM_NAME = "HopeEXP2026"
RUN_NUM = 1

PRED_JSONL = os.path.join(OUT_DIR, "submission.jsonl")
ZIP_PATH   = os.path.join(OUT_DIR, f"{TEAM_NAME}_{RUN_NUM}.zip")

EMOTIONS_ALL = ['sadness','joy','love','anger','fear','surprise',"Nuetral/unclear"]

# -------------------------
# IO
# -------------------------
def load_jsonl(path):
    recs = []
    with open(path,"r",encoding="utf-8") as f:
        for line in f:
            recs.append(json.loads(line))
    return pd.DataFrame(recs)

def json_safe_lang(v):
    if v is None:
        return None
    try:
        if pd.isna(v):
            return None
    except:
        pass
    return str(v)

def build_text(df):
    return (df["title"].fillna("")+"\n\n"+df["selftext"].fillna("")).str.strip()

def normalize_emotions_list(x):

    if isinstance(x,list):
        labs=[str(z).strip() for z in x if str(z).strip()]
    else:
        labs=[str(x).strip()] if x else []

    all_low={a.lower():a for a in EMOTIONS_ALL}

    out=[]
    for e in labs:
        if e in EMOTIONS_ALL:
            out.append(e)
        else:
            out.append(all_low.get(e.lower(),"Nuetral/unclear"))

    if not out:
        out=["Nuetral/unclear"]

    if "Nuetral/unclear" in out and len(out)>1:
        out=[z for z in out if z!="Nuetral/unclear"]

    return list(dict.fromkeys(out))

# -------------------------
# RULE spans
# -------------------------

HOPE_CUES=[r"\bi hope\b",r"\bhope\b",r"\bwish\b",r"\bhopefully\b"]

NEG_CUES={"not","never","no"}

WORLD_KWS={"weather","economy","doctor","clinic","therapy"}

OTHER_KWS={"they","landlord","boss","manager","company"}

def extract_spans_rule(text,max_spans=3):

    t=text.strip()
    low=t.lower()

    cue_end=None
    for pat in HOPE_CUES:
        m=re.search(pat,low)
        if m:
            cue_end=m.end()
            break

    if cue_end is None:
        return []

    tail=t[cue_end:]

    parts=re.split(r"\b(?:and|but|so|that)\b|[.;\n]",tail)

    parts=[p.strip(" .,!?") for p in parts if len(p)>2]

    return parts[:max_spans]

def stance_rule(span):

    s=span.lower()

    for c in NEG_CUES:
        if c in s:
            return "Avoided"

    return "Desired"

def actor_rule(span):

    s=span.lower()

    if re.search(r"\b(i|my|me)\b",s):
        return "Self"

    if any(k in s for k in WORLD_KWS):
        return "World/System"

    if any(k in s for k in OTHER_KWS):
        return "Other"

    return "Unclear"

def rule_span_items(text,label):

    if label in ['General Hope','Hopelessness','Not Hope',
                 'Realistic Hope','Sarcastic Hope','Unrealistic Hope']:
        return []

    spans=extract_spans_rule(text)

    return [
        {"span":sp,
         "outcome_stance":stance_rule(sp),
         "actor":actor_rule(sp)}
        for sp in spans
    ]

# -------------------------
# LOAD DATA
# -------------------------

df_train=load_jsonl(TRAIN_GOLD_JSONL)
df_dev=load_jsonl(DEV_UNLAB_JSONL)

for df in (df_train,df_dev):

    if "lang" not in df.columns:
        df["lang"]=None

    df["title"]=df["title"].fillna("")
    df["selftext"]=df["selftext"].fillna("")
    df["text"]=build_text(df)

df_train["trigger_emotions"]=df_train["trigger_emotions"].apply(normalize_emotions_list)

# -------------------------
# TFIDF + SVD
# -------------------------

vec=TfidfVectorizer(max_features=2000,min_df=2,max_df=0.9)

Xtr=vec.fit_transform(df_train["text"])
Xdv=vec.transform(df_dev["text"])

svd=TruncatedSVD(n_components=300)

Xtr_reduced=svd.fit_transform(Xtr)
Xdv_reduced=svd.transform(Xdv)

train_df=pd.DataFrame(Xtr_reduced)
dev_df=pd.DataFrame(Xdv_reduced)

# =====================================================
# Task A — primary_label
# =====================================================

train_A=train_df.copy()
train_A["label"]=df_train["primary_label"].astype(str)

predictor_A=TabularPredictor(
    label="label",
    path=os.path.join(OUT_DIR,"model_A")
).fit(
    train_A,
    presets="best_quality"
)

pred_A=predictor_A.predict(dev_df).tolist()

# =====================================================
# Task B — multi-label emotions
# =====================================================

mlb=MultiLabelBinarizer(classes=EMOTIONS_ALL)

Ytr=mlb.fit_transform(df_train["trigger_emotions"])

emotion_models={}

pred_B=[]

for i,emotion in enumerate(EMOTIONS_ALL):

    train_B=train_df.copy()
    train_B["label"]=Ytr[:,i]

    predictor=TabularPredictor(
        label="label",
        path=os.path.join(OUT_DIR,f"emotion_{emotion}")
    ).fit(
        train_B,
        presets="best_quality"
    )

    emotion_models[emotion]=predictor

pred_matrix=[]

for emotion in EMOTIONS_ALL:

    model=emotion_models[emotion]

    pred=model.predict(dev_df).values

    pred_matrix.append(pred)

pred_matrix=np.array(pred_matrix).T

for row in pred_matrix:

    labs=[EMOTIONS_ALL[i] for i,v in enumerate(row) if int(v)==1]

    if not labs:
        labs=["Nuetral/unclear"]

    if "Nuetral/unclear" in labs and len(labs)>1:
        labs=[z for z in labs if z!="Nuetral/unclear"]

    pred_B.append(labs)

# -------------------------
# WRITE JSONL
# -------------------------

with open(PRED_JSONL,"w",encoding="utf-8") as f:

    for rid,lang,title,selftext,text,plab,ems in zip(
        df_dev["row_id"],
        df_dev["lang"],
        df_dev["title"],
        df_dev["selftext"],
        df_dev["text"],
        pred_A,
        pred_B
    ):

        spans=rule_span_items(text,str(plab))

        obj={
            "row_id":int(rid),
            "lang":json_safe_lang(lang),
            "title":title,
            "selftext":selftext,
            "primary_label":str(plab),
            "trigger_emotions":ems,
            "span_annotations":spans
        }

        f.write(json.dumps(obj,ensure_ascii=False)+"\n")

print("✅ JSONL created")

# -------------------------
# ZIP
# -------------------------

with zipfile.ZipFile(ZIP_PATH,"w",compression=zipfile.ZIP_DEFLATED) as z:
    z.write(PRED_JSONL,arcname=os.path.basename(PRED_JSONL))

print("✅ ZIP created:",ZIP_PATH)

/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.1.1
Python Version:     3.8.20
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP PREEMPT_DYNAMIC Thu Jun  5 18:30:46 UTC 2025
CPU Count:          20
Memory Avail:       21.97 GB / 23.38 GB (93.9%)
Disk Space Avail:   463.52 GB / 1862.13 GB (24.9%)
Presets specified: ['best_quality']
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=8, num_bag_sets=1
DyStack is enabled (dynamic_stacking=True). AutoGluon will try to determ

[1000]	valid_set's multi_error: 0.164815


	0.7996	 = Validation score   (accuracy)
	131.64s	 = Training   runtime
	0.11s	 = Validation runtime
Fitting model: LightGBM_BAG_L1 ... Training model for up to 464.59s of the 764.06s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy


[1000]	valid_set's multi_error: 0.190741
[1000]	valid_set's multi_error: 0.238889
[1000]	valid_set's multi_error: 0.225926
[1000]	valid_set's multi_error: 0.188889


	0.7869	 = Validation score   (accuracy)
	140.44s	 = Training   runtime
	0.1s	 = Validation runtime
Fitting model: RandomForestGini_BAG_L1 ... Training model for up to 322.9s of the 622.38s of remaining time.
	0.7183	 = Validation score   (accuracy)
	1.24s	 = Training   runtime
	0.24s	 = Validation runtime
Fitting model: RandomForestEntr_BAG_L1 ... Training model for up to 321.0s of the 620.48s of remaining time.
	0.7151	 = Validation score   (accuracy)
	1.79s	 = Training   runtime
	0.22s	 = Validation runtime
Fitting model: CatBoost_BAG_L1 ... Training model for up to 318.66s of the 618.14s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy
	Ran out of time, early stopping on iteration 332.
	Ran out of time, early stopping on iteration 318.
	Ran out of time, early stopping on iteration 387.
	Ran out of time, early stopping on iteration 298.
	Ran out of time, early stopping on iteration 302.
	Ran out of time, early stopping on ite

...

[1000]	valid_set's binary_error: 0.14827


	0.854	 = Validation score   (accuracy)
	6.59s	 = Training   runtime
	0.02s	 = Validation runtime
Fitting model: NeuralNetTorch_r22_BAG_L1 ... Training model for up to 1024.3s of the 1922.28s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy
/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/autogluon/tabular/models/tabular_nn/torch/tabular_nn_torch.py:411: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded vi

[1000]	valid_set's binary_error: 0.149918


	0.8536	 = Validation score   (accuracy)
	19.39s	 = Training   runtime
	0.01s	 = Validation runtime
Fitting model: NeuralNetTorch_r86_BAG_L1 ... Training model for up to 104.15s of the 1002.13s of remaining time.
	Fitting 8 child models (S1F1 - S1F8) | Fitting with SequentialLocalFoldFittingStrategy
/home/wangkongqiang/miniconda3/envs/HopeEXP2026/lib/python3.8/site-packages/autogluon/tabular/models/tabular_nn/torch/tabular_nn_torch.py:411: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded 

✅ JSONL created
✅ ZIP created: Hope-EXP datasets/baseline_AutoGluon_TruncatedSVD_submission/HopeEXP2026_1.zip
